In [ ]:
# Cell 1: Environment setup, imports, and global configuration.

%matplotlib inline

import json
import os
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap

from IPython.display import display


from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression  
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC 
from pathlib import Path 
from scipy import stats 
from scipy.stats import t
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportion_confint

# Suppress non-critical warnings to keep notebook output clean.
warnings.filterwarnings("ignore")

# Set a fixed seed to make results reproducible.
SEED = 42
np.random.seed(SEED)


In [ ]:
# Cell 2: Define dataset path, feature sets, target variable, and output directory structure.

# File path

DATA_PATH = Path(os.environ.get("CLABSI_DATA_PATH", "data/clabsi_dataset.xlsx"))

# Target and features
target = "CLABSI"

categorical_features = [
    "Seasonofinsertion",
    "Sex",
    "ICUSource",
    "HospitalSource",
    "Smoking Status",
    "Admissionin3months",
    "Immunosuppressed",
    "INV",
    "Inotropes",
    "Diabetes",
    "Plannedadmission",
    "RenalRep",
    "Thrombopro",
    "HospitalizationType",
    "ICU Type",
    "Immune_Disease",
    "Metastases",
    "Chronic_respiratory",
    "Chronic_Cardiovascular",
    "Chronic_Renal_failure",
    "Antibiotictype_yes",
]

numerical_features = [
    "PreICULOS",
    "Antibiotictype",
    "BMI",
    "TPNDuration",
    "BS",
    "BUN",
    "PT",
    "RDW",
    "MCHC",
    "MCH",
    "MCV",
    "Neut",
    "Lymphocyte",
    "Age",
    "Frailty",
    "Temp",
    "HR",
    "MAP",
    "RR",
    "Na",
    "K_1",
    "HCO3",
    "Creat",
    "Albumin",
    "HCT",
    "HMG",
    "WBC",
    "PLT",
    "PH",
    "GCS",
    "APACHEII",
    "Bilirubin",
]

all_features = categorical_features + numerical_features

# Output directories (shared timestamp for main model and LOIO)
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"journal_ready_svm_clabsi_{RUN_TIMESTAMP}")

MAIN_OUTPUT_DIR = OUTPUT_DIR / "main_model"
FIG_DIR = MAIN_OUTPUT_DIR / "figures"
TABLE_DIR = MAIN_OUTPUT_DIR / "tables"
MODEL_INFO_DIR = MAIN_OUTPUT_DIR / "model_info"

LOIO_OUTPUT_DIR = OUTPUT_DIR / "loio"
LOIO_FIG_DIR = LOIO_OUTPUT_DIR / "figures"
LOIO_TABLE_DIR = LOIO_OUTPUT_DIR / "tables"
LOIO_MODEL_INFO_DIR = LOIO_OUTPUT_DIR / "model_info"

for output_path in [
    FIG_DIR,
    TABLE_DIR,
    MODEL_INFO_DIR,
    LOIO_FIG_DIR,
    LOIO_TABLE_DIR,
    LOIO_MODEL_INFO_DIR,
]:
    output_path.mkdir(parents=True, exist_ok=True)

print("Shared run output structure created successfully.")
print(f"Run timestamp: {RUN_TIMESTAMP}")
print(f"Shared root directory: {OUTPUT_DIR.resolve()}")
print(f"Main-model directory: {MAIN_OUTPUT_DIR.resolve()}")
print(f"LOIO directory: {LOIO_OUTPUT_DIR.resolve()}")


In [ ]:
# Cell 3: Define shared preprocessing, tuning, calibration, and evaluation for the main model and LOIO

TOP_K_FEATURES = 15

CALIBRATION_METHOD = "sigmoid"  

# Shared hyperparameter grid 
param_grid = {
    "classifier__C": [0.01, 0.1, 1, 5],
    "classifier__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 1.5},
        {0: 1, 1: 2},
        {0: 1, 1: 3},
    ],
}

# Shared performance and calibration metric functions.
def calculate_metrics(y_true, y_proba, threshold=0.5):
    """Calculate discrimination, classification, and Brier-score metrics."""
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_pred = (y_proba >= threshold).astype(int)

    cm = confusion_matrix(y_true, y_pred)
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        # Handle single-class evaluation subsets.
        tn = fp = fn = tp = 0
        if len(np.unique(y_true)) == 1:
            if y_true[0] == 0:
                tn = len(y_true)
            else:
                tp = len(y_true)

    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_PPV": precision_score(y_true, y_pred, zero_division=0),
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "NPV": tn / (tn + fn) if (tn + fn) > 0 else 0.0,
        "F1_score": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "PR_AUC": average_precision_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "Brier": brier_score_loss(y_true, y_proba),
    }
    return metrics


def calibration_slope_intercept(y_true, y_proba):
    """Estimate calibration intercept and slope on the logit scale."""
    if len(np.unique(y_true)) < 2:
        return np.nan, np.nan

    eps = 1e-6
    y_proba_clipped = np.clip(y_proba, eps, 1 - eps)
    logit_p = np.log(y_proba_clipped / (1 - y_proba_clipped)).reshape(-1, 1)

    try:
        cal_model = LogisticRegression(penalty=None, solver="lbfgs").fit(logit_p, y_true)
        return cal_model.intercept_[0], cal_model.coef_[0][0]
    except Exception:
        return np.nan, np.nan


def calculate_all_metrics(y_true, y_proba, threshold=0.5):
    """Calculate classification and calibration metrics."""
    metrics = calculate_metrics(y_true, y_proba, threshold)
    cal_intercept, cal_slope = calibration_slope_intercept(y_true, y_proba)
    metrics["Calibration_intercept"] = cal_intercept
    metrics["Calibration_slope"] = cal_slope
    return metrics


# Shared preprocessing and linear-SVM pipeline functions.
def _make_preprocessor(cat_cols, num_cols):
    """Build preprocessing from predefined categorical and numerical feature lists."""
    try:
        cat_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
    except TypeError:
        cat_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[("scaler", StandardScaler())]), num_cols),
            ("cat", cat_encoder, cat_cols),
        ],
        remainder="drop",
    )


def _linear_svm_pipeline(cat_cols, num_cols, seed):
    """Build the shared preprocessing and linear-SVM pipeline."""
    return Pipeline(
        steps=[
            ("preprocessor", _make_preprocessor(cat_cols, num_cols)),
            ("classifier", SVC(kernel="linear", probability=False, random_state=seed)),
        ]
    )

def _decision_function_auc_scorer(estimator, X, y):
    """Calculate AUROC from raw SVM decision-function scores."""
    scores = estimator.decision_function(X)
    return roc_auc_score(y, scores)


# Shared three-stage tuning and SHAP-based feature-selection routine.
def select_features_and_tune(
    X_train,
    y_train,
    categorical_features,
    numerical_features,
    param_grid,
    seed=SEED,
    top_k=TOP_K_FEATURES,
    cv_splits=5,
    scoring="roc_auc",
):
    """
    Stage 1: Tune hyperparameters using all features on the training set.
    Stage 2: Select Top-15 features using SHAP values from the Stage-1 model.
    Stage 3: Ttune hyperparameters again using selected features.
    Validation and test set are separated from these stages.
    """
    cat_avail = [c for c in categorical_features if c in X_train.columns]
    num_avail = [c for c in numerical_features if c in X_train.columns]

    n_splits = max(2, min(cv_splits, int(np.min(np.bincount(np.asarray(y_train))))))
    inner_cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # Stage 1 #
    full_pipeline = _linear_svm_pipeline(cat_avail, num_avail, seed)
    full_search = GridSearchCV(
        estimator=full_pipeline,
        param_grid=param_grid,
        scoring=_decision_function_auc_scorer,
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
    )
    full_search.fit(X_train[cat_avail + num_avail], y_train)
    full_best_model = full_search.best_estimator_

    # Stage 2 #.
    fitted_preprocessor = full_best_model.named_steps["preprocessor"]
    svm_step = full_best_model.named_steps["classifier"]

    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
    except Exception:
        feature_names = np.array(
            num_avail
            + list(
                fitted_preprocessor.named_transformers_["cat"].get_feature_names_out(cat_avail)
            )
        )
    feature_names = np.array(feature_names)

    X_train_processed = fitted_preprocessor.transform(X_train[cat_avail + num_avail])
    X_train_processed_df = pd.DataFrame(
        X_train_processed, columns=feature_names, index=X_train.index
    )

    background_size = min(100, X_train_processed_df.shape[0])
    X_background = shap.sample(X_train_processed_df, background_size, random_state=seed)

    explainer = shap.LinearExplainer(svm_step, X_background)
    shap_values_result = explainer(X_train_processed_df)
    shap_values_array = (
        shap_values_result.values if hasattr(shap_values_result, "values") else shap_values_result
    )
    if np.ndim(shap_values_array) == 3:
        shap_values_array = shap_values_array[:, :, 1]

    importance_rows = []
    for i, f_name in enumerate(feature_names):
        clean_name = f_name.replace("num__", "").replace("cat__", "")
        orig_var = clean_name
        for cat_var in cat_avail:
            if clean_name.startswith(cat_var + "_"):
                orig_var = cat_var
                break
        importance_rows.append(
            {"Original_Variable": orig_var, "Abs_SHAP": np.abs(shap_values_array[:, i]).mean()}
        )

    shap_agg = (
        pd.DataFrame(importance_rows)
        .groupby("Original_Variable")["Abs_SHAP"]
        .sum()
        .sort_values(ascending=False)
    )
    selected_features = shap_agg.head(top_k).index.tolist()

    # Stage 3 #
    sel_cat = [c for c in categorical_features if c in selected_features]
    sel_num = [c for c in numerical_features if c in selected_features]

    final_pipeline_template = _linear_svm_pipeline(sel_cat, sel_num, seed)
    final_search = GridSearchCV(
        estimator=final_pipeline_template,
        param_grid=param_grid,
        scoring=_decision_function_auc_scorer,
        cv=inner_cv,
        n_jobs=-1,
        refit=True,
    )
    final_search.fit(X_train[selected_features], y_train)

    return {
        "final_pipeline": final_search.best_estimator_,
        "selected_features": selected_features,
        "full_feature_best_params": full_search.best_params_,
        "final_best_params": final_search.best_params_,
        "shap_importance": shap_agg,
    }


# Calibration functions for model probabilities
def fit_calibrator(pipeline, X_cal, y_cal, selected_features, method=CALIBRATION_METHOD):
    """Fit a score-to-probability calibration model on a validation partition."""
    raw_scores = pipeline.decision_function(X_cal[selected_features]).reshape(-1, 1)
    y_cal_arr = np.asarray(y_cal)

    if method == "isotonic":
        calibrator = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        calibrator.fit(raw_scores.ravel(), y_cal_arr)
    elif method == "sigmoid":
        calibrator = LogisticRegression(solver="lbfgs")
        calibrator.fit(raw_scores, y_cal_arr)
    else:
        raise ValueError(f"Unknown calibration method: {method!r}. Use 'sigmoid' or 'isotonic'.")

    return calibrator


def apply_calibrator(pipeline, calibrator, X, selected_features, method=CALIBRATION_METHOD):
    """Convert raw SVM scores to calibrated event probabilities."""
    raw_scores = pipeline.decision_function(X[selected_features]).reshape(-1, 1)

    if method == "isotonic":
        proba = calibrator.predict(raw_scores.ravel())
    else:
        proba = calibrator.predict_proba(raw_scores)[:, 1]

    return np.clip(np.asarray(proba, dtype=float), 0.0, 1.0)


def tune_threshold_on_val(pipeline, X_val, selected_features, y_val, calibrator, method=CALIBRATION_METHOD):
    """Select the probability threshold that maximizes F1 on validation data."""
    proba = apply_calibrator(pipeline, calibrator, X_val, selected_features, method=method)
    precisions, recalls, thresholds = precision_recall_curve(y_val, proba)
    f1_scores = np.divide(
        2 * (precisions * recalls),
        (precisions + recalls),
        out=np.zeros_like(precisions),
        where=(precisions + recalls) != 0,
    )
    best_idx = np.argmax(f1_scores[:-1]) if len(thresholds) > 0 else 0
    best_threshold = float(thresholds[best_idx]) if len(thresholds) > 0 else 0.5
    return best_threshold, proba


def calibrate_and_tune_threshold(pipeline, X_val, y_val, selected_features, method=CALIBRATION_METHOD):
    """
    Fit the calibration model and choose the threshold that maximizes the F1 score
    using the validation data.
    Because the same validation set is used for both calibration and threshold selection,
    the estimated threshold-dependent performance may be slightly optimistic.
    """
    calibrator = fit_calibrator(pipeline, X_val, y_val, selected_features, method=method)
    best_threshold, val_proba = tune_threshold_on_val(
        pipeline, X_val, selected_features, y_val, calibrator, method=method
    )
    return calibrator, best_threshold, val_proba


print(
    "Unified pipeline loaded (TOP_K_FEATURES = {}, CALIBRATION_METHOD = '{}').".format(
        TOP_K_FEATURES, CALIBRATION_METHOD
    )
)


In [ ]:
# ==============================================================================
# Cell 4: Load dataset, validate required columns, and audit Patient_Identifier
# ==============================================================================

# Metadata and grouping columns for LOIO
group_column = "ICU Name"
patient_id_col = "Patient_Identifier"

# Load dataset
df = pd.read_excel(DATA_PATH)
print(f"Original dataset shape: {df.shape}")

# Validate required columns
required_columns = all_features + [target, group_column, patient_id_col]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {', '.join(missing_columns)}")

# Keep only analysis features and required metadata
df = df[required_columns].copy()

# Validate Patient_Identifier completeness
missing_patient_ids = int(df[patient_id_col].isna().sum())
if missing_patient_ids > 0:
    raise ValueError(
        f"Data Integrity Error: {missing_patient_ids} records have missing '{patient_id_col}'."
    )

df[patient_id_col] = df[patient_id_col].astype(str).str.strip()
empty_patient_ids = int(df[patient_id_col].isin(["", "nan", "None"]).sum())
if empty_patient_ids > 0:
    raise ValueError(
        f"Data Integrity Error: {empty_patient_ids} records have empty '{patient_id_col}' values."
    )

# Remove missing values across analysis variables
before_drop_shape = df.shape
df = df.dropna().reset_index(drop=True)
after_drop_shape = df.shape

print(f"Shape after column selection: {before_drop_shape}")
print(f"Shape after dropping missing values: {after_drop_shape}")
print(f"Total rows removed because of missingness: {before_drop_shape[0] - after_drop_shape[0]}")

# 5. Patient_Identifier uniqueness check
duplicate_patient_mask = df[patient_id_col].duplicated(keep=False)
duplicate_patient_rows = df.loc[duplicate_patient_mask, [patient_id_col, group_column]].copy()

n_duplicate_patient_rows = len(duplicate_patient_rows)
n_duplicate_patient_ids = int(duplicate_patient_rows[patient_id_col].nunique())

if n_duplicate_patient_rows > 0:
    duplicate_patient_summary = (
        duplicate_patient_rows.groupby(patient_id_col, dropna=False)
        .agg(
            Number_of_records=(patient_id_col, "size"),
            ICU_values=(group_column, lambda values: sorted(values.astype(str).unique().tolist())),
        )
        .reset_index()
        .sort_values("Number_of_records", ascending=False)
    )

    print("Duplicate Patient_Identifier values detected:")
    display(duplicate_patient_summary.head(20))

    raise ValueError(
        f"Error: Repeated Patient_Identifier values detected for "
        f"{n_duplicate_patient_ids} patients across {n_duplicate_patient_rows} rows."
    )

print(
    f"✓ Patient_Identifier uniqueness confirmed: {df[patient_id_col].nunique()} "
    f"unique identifiers for {len(df)} rows."
)
print(
    f"ICUs available for LOIO: {df[group_column].nunique()} -> {list(df[group_column].unique())}"
)

#Summary 
print("\nDataset info:")
df.info()


In [ ]:
# ==============================================================================
# Cell 5: Separate features from metadata and check for data leakage
# ==============================================================================

# Make sure all required variables and columns exist
required_objects = ["df", "all_features", "target", "group_column", "patient_id_col"]
missing_objects = [obj for obj in required_objects if obj not in globals()]
if missing_objects:
    raise NameError(f"Missing required objects: {', '.join(missing_objects)}. Please run previous cells first.")

required_columns = all_features + [target, group_column, patient_id_col]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise KeyError(f"These required columns are missing in the dataset: {', '.join(missing_columns)}")

if len(required_columns) != len(set(required_columns)):
    duplicates = [col for col in set(required_columns) if required_columns.count(col) > 1]
    raise ValueError(f"Duplicate column names found: {', '.join(duplicates)}")

# Prepare the final clean dataset (remove infinite and missing values)
data = df.loc[:, required_columns].copy()
data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

if data.empty:
    raise ValueError("No records left after cleaning missing and infinite values.")

#  Double-check that each patient appears only once
data[patient_id_col] = data[patient_id_col].astype(str).str.strip()
if data[patient_id_col].isin(["", "nan", "None"]).any():
    raise ValueError(f"Found empty or invalid values in '{patient_id_col}'.")

if data[patient_id_col].duplicated().any():
    duplicated_ids = data.loc[data[patient_id_col].duplicated(), patient_id_col].unique().tolist()
    raise ValueError(
        f"Data Integrity Error: Found duplicate patients in the dataset "
        f"({len(duplicated_ids)} duplicate IDs). Examples: {duplicated_ids[:5]}"
    )

# 4. Separate model features (X), target outcome (y), and tracking metadata
X = data.loc[:, all_features].copy()
y = data[target].astype(int).copy()

# Keep ICU and Patient ID separately for validation and auditing (never as predictors)
icu_groups = data[group_column].copy()
patient_groups = data[patient_id_col].copy()

# Prevent data leakage: ensure metadata never enters the feature matrix X
metadata_columns = {target, group_column, patient_id_col}
leakage_columns = metadata_columns.intersection(X.columns)
if leakage_columns:
    raise ValueError(f"Data Leakage Risk: Found metadata/target columns inside X: {sorted(leakage_columns)}")

if X.shape[0] != y.shape[0] or not X.index.equals(y.index):
    raise RuntimeError("Mismatch in row count or indexing between X and y.")

if patient_groups.nunique() != len(patient_groups):
    raise RuntimeError("Patient uniqueness check failed: duplicate IDs exist.")

print("✓ Feature matrix (X) and target (y) are ready.")
print("✓ Leakage check passed: Target, ICU Name, and Patient ID are completely excluded from X.")
print(f"✓ Total unique patients: {patient_groups.nunique()}")
print(f"X shape: {X.shape}, Target distribution: {dict(y.value_counts())}")
print(f"\nICU breakdown ({icu_groups.nunique()} ICUs):")
print(icu_groups.value_counts())



In [ ]:
# ==============================================================================
# Cell 6: Split data into training, validation, and test sets (60 / 20 / 20)
# ==============================================================================

# Check that all required inputs are present
required_objects = ["X", "y", "patient_groups", "SEED", "TABLE_DIR"]
missing_objects = [obj for obj in required_objects if obj not in globals()]
if missing_objects:
    raise NameError(f"Missing required inputs: {', '.join(missing_objects)}. Please run earlier cells first.")

# Basic safety checks before splitting
if not (len(X) == len(y) == len(patient_groups)):
    raise ValueError("Lengths of X, y, and patient_groups do not match.")

if not X.index.equals(y.index) or not y.index.equals(patient_groups.index):
    raise ValueError("Index mismatch between X, y, or patient_groups.")

if y.nunique() != 2:
    raise ValueError(f"Expected binary outcome (0 and 1), but found classes: {sorted(y.unique().tolist())}")

if y.value_counts().min() < 5:
    raise ValueError("Too few positive or negative cases to perform stratified splitting.")

# Step 1: Separate the held-out test set (20%) from the development set (80%)
record_indices = np.arange(len(y))
idx_dev, idx_test = train_test_split(
    record_indices,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

# Step 2: Split development set into train (60% total) and validation (20% total)
idx_train, idx_val = train_test_split(
    idx_dev,
    test_size=0.25,
    stratify=y.iloc[idx_dev],
    random_state=SEED,
)

# Build features, labels, and tracking IDs for each split
X_train, y_train = X.iloc[idx_train].copy(), y.iloc[idx_train].copy()
X_val, y_val = X.iloc[idx_val].copy(), y.iloc[idx_val].copy()
X_test, y_test = X.iloc[idx_test].copy(), y.iloc[idx_test].copy()

# Keep patient IDs strictly for tracking and auditing 
patient_train = patient_groups.iloc[idx_train].copy()
patient_val = patient_groups.iloc[idx_val].copy()
patient_test = patient_groups.iloc[idx_test].copy()

# Ensure no patient or row is shared across sets
train_idx_set, val_idx_set, test_idx_set = set(idx_train), set(idx_val), set(idx_test)
if (train_idx_set & val_idx_set) or (train_idx_set & test_idx_set) or (val_idx_set & test_idx_set):
    raise RuntimeError("Data Leakage: Overlapping rows found between splits!")

if len(train_idx_set | val_idx_set | test_idx_set) != len(y):
    raise RuntimeError("Split error: Total split records do not match original dataset size.")

train_pts, val_pts, test_pts = set(patient_train), set(patient_val), set(patient_test)
if (train_pts & val_pts) or (train_pts & test_pts) or (val_pts & test_pts):
    raise ValueError("Patient Leakage: The same patient appears in more than one split!")

print("✓ Data splitting completed successfully.")
print("✓ Leakage check passed: Train, Validation, and Test sets are completely independent.")

# Summarize class distribution across splits
split_summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test", "Total"],
    "N": [len(y_train), len(y_val), len(y_test), len(y)],
    "Unique_Patients": [patient_train.nunique(), patient_val.nunique(), patient_test.nunique(), patient_groups.nunique()],
    "CLABSI_Positive": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum()), int(y.sum())],
    "CLABSI_Negative": [int((y_train == 0).sum()), int((y_val == 0).sum()), int((y_test == 0).sum()), int((y == 0).sum())],
    "Event_Rate": [y_train.mean(), y_val.mean(), y_test.mean(), y.mean()],
})
split_summary["Event_Rate"] = split_summary["Event_Rate"].round(4)
display(split_summary)

# Save summary table
split_summary_path = TABLE_DIR / "data_split_summary.csv"
split_summary.to_csv(split_summary_path, index=False)
print(f"✓ Split summary saved to: {split_summary_path}")


In [ ]:
# ==============================================================================
# Cell 7: Main Model — Feature selection, hyperparameter tuning, and provenance logging
# ==============================================================================

# Run the  3-stage pipeline on the Train set:
# Stage 1: Hyperparameter tuning using all features
# Stage 2: Feature importance using SHAP values
# Stage 3: Tuning the model again  only with the top-ranked features
print("Training main model: running 3-stage feature selection and tuning on Train set...")

pipeline_result = select_features_and_tune(
    X_train,
    y_train,
    categorical_features,
    numerical_features,
    param_grid,
    seed=SEED,
    top_k=TOP_K_FEATURES,
)

# Pipeline, selected features, and tuning parameters
final_pipeline = pipeline_result["final_pipeline"]
top_15_features = pipeline_result["selected_features"]
full_feature_best_params = pipeline_result["full_feature_best_params"]
final_best_params = pipeline_result["final_best_params"]
shap_importance = pipeline_result["shap_importance"]

# Tuning summary
print(f"\nBest hyperparameters using all features (Stage 1): {full_feature_best_params}")
print(f"Final hyperparameters after selecting Top-{len(top_15_features)} features (Stage 3): {final_best_params}")

# Display the final selected features
print(f"\n--- Top {len(top_15_features)} Selected Features ---")
for rank, feature_name in enumerate(top_15_features, start=1):
    print(f"{rank}. {feature_name}")

top_features_path = MODEL_INFO_DIR / "top_15_features.json"
with open(top_features_path, "w", encoding="utf-8") as f:
    json.dump(top_15_features, f, indent=4)
print(f"\n✓ Saved top features to: {top_features_path}")

stage1_params_path = MODEL_INFO_DIR / "full_feature_best_parameters.json"
with open(stage1_params_path, "w", encoding="utf-8") as f:
    json.dump(full_feature_best_params, f, indent=4, default=str)
print(f"✓ Saved Stage 1 best hyperparameters to: {stage1_params_path}")

final_params_path = MODEL_INFO_DIR / "final_best_parameters.json"
with open(final_params_path, "w", encoding="utf-8") as f:
    json.dump(final_best_params, f, indent=4, default=str)
print(f"✓ Saved final best hyperparameters to: {final_params_path}")


In [ ]:
# ==============================================================================
# Cell 8: Plot and save SHAP feature importance (Training set)
# ==============================================================================

shap_agg_df = shap_importance.reset_index()
shap_agg_df.columns = ["Original_Variable", "Abs_SHAP"]
top_shap_df = shap_agg_df.head(len(top_15_features))
fig, ax = plt.subplots(figsize=(10, 6), layout="tight")
sns.barplot(
    data=top_shap_df,
    x="Abs_SHAP",
    y="Original_Variable",
    hue="Original_Variable",
    palette="Blues_r",
    edgecolor="0.2",
    linewidth=0.8,
    legend=False,
    ax=ax,
)
ax.set_title("Feature Importance: Mean Absolute SHAP Value (Train Set)", pad=15, weight="bold")
ax.set_xlabel("Mean Absolute SHAP Value", labelpad=10)
ax.set_ylabel("Feature Name", labelpad=10)
ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="x", linestyle="--", alpha=0.5)
png_path = FIG_DIR / "top_15_features_importance.png"
pdf_path = FIG_DIR / "top_15_features_importance.pdf"

fig.savefig(png_path, dpi=600, bbox_inches="tight")
fig.savefig(pdf_path, bbox_inches="tight")
plt.show()

print(f"✓ Feature importance plots saved to:\n  - {png_path}\n  - {pdf_path}")


In [ ]:
# ==============================================================================
# Cell 9: Calibration & decision threshold tuning (Validation set only)
# ==============================================================================

# Calibrate probabilities and tune threshold on Validation set(based on F1)
final_calibrator, best_threshold, y_val_proba = calibrate_and_tune_threshold(
    final_pipeline, X_val, y_val, top_15_features, method=CALIBRATION_METHOD
)

# Apply fitted calibrator to Train and Test sets
y_train_proba = apply_calibrator(final_pipeline, final_calibrator, X_train, top_15_features, method=CALIBRATION_METHOD)
y_test_proba = apply_calibrator(final_pipeline, final_calibrator, X_test, top_15_features, method=CALIBRATION_METHOD)

print(f"Calibration Method: {CALIBRATION_METHOD} (Fit on Validation)")
print(f"Optimal Decision Threshold: {best_threshold:.4f}")
threshold_info = {
    "optimal_threshold": float(best_threshold),
    "calibration_method": CALIBRATION_METHOD,
    "calibration_fit_on": "Validation_only",
}
with open(MODEL_INFO_DIR / "final_threshold.json", "w", encoding="utf-8") as f:
    json.dump(threshold_info, f, indent=4)


In [ ]:
# ==============================================================================
# Cell 10:Performance evaluation across all splits
# ==============================================================================

train_metrics = calculate_all_metrics(y_train, y_train_proba, best_threshold)
val_metrics = calculate_all_metrics(y_val, y_val_proba, best_threshold)
test_metrics = calculate_all_metrics(y_test, y_test_proba, best_threshold)

metrics_comparison_df = pd.DataFrame([
    {"Dataset": "Train", **train_metrics},
    {"Dataset": "Validation", **val_metrics},
    {"Dataset": "Test", **test_metrics},
])
display(metrics_comparison_df.round(4))
metrics_table_path = TABLE_DIR / "train_validation_test_metrics_comparison.csv"
metrics_comparison_df.to_csv(metrics_table_path, index=False)
print(f"✓ Metrics comparison saved to: {metrics_table_path}")


In [ ]:
# ==============================================================================
# Cell 11: Export Test-set point estimates
# ==============================================================================

test_metrics_df = (
    pd.DataFrame.from_dict(test_metrics, orient="index", columns=["Value"])
    .reset_index()
    .rename(columns={"index": "Metric"})
)
test_metrics_df["Value"] = test_metrics_df["Value"].round(4)

print("=== Final Test-Set Evaluation: Point Estimates ===")
display(test_metrics_df)
test_metrics_path = TABLE_DIR / "test_metrics_point_estimates.csv"
test_metrics_df.to_csv(test_metrics_path, index=False)
print(f"✓ Saved to: {test_metrics_path}")


In [ ]:
# ==============================================================================
# Cell 12: Stratified bootstrap 95% confidence intervals (Test set)
# ==============================================================================

def bootstrap_metrics_ci(
    y_true,
    y_proba,
    threshold,
    point_estimates,
    n_bootstraps=2000,
    ci=95,
    random_state=SEED,
):
    """Calculate percentile bootstrap CIs using stratified resampling with replacement."""
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=float)

    if len(y_true) != len(y_proba):
        raise ValueError("y_true and y_proba must have the same length.")

    rng = np.random.default_rng(random_state)
    pos_idx = np.flatnonzero(y_true == 1)
    neg_idx = np.flatnonzero(y_true == 0)

    bootstrap_results = []
    for _ in range(n_bootstraps):
        s_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        s_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([s_pos, s_neg])
        rng.shuffle(sampled_idx)

        bootstrap_results.append(
            calculate_all_metrics(
                y_true=y_true[sampled_idx],
                y_proba=y_proba[sampled_idx],
                threshold=threshold,
            )
        )

    bootstrap_df = pd.DataFrame(bootstrap_results)
    lower_p = (100 - ci) / 2
    upper_p = 100 - lower_p

    ci_rows = []
    for metric, pt_val in point_estimates.items():
        if metric not in bootstrap_df.columns:
            continue
        vals = bootstrap_df[metric].dropna()
        if vals.empty:
            lo, hi = np.nan, np.nan
            formatted = f"{pt_val:.3f} (NA-NA)"
        else:
            lo, hi = np.percentile(vals, lower_p), np.percentile(vals, upper_p)
            formatted = f"{pt_val:.3f} ({lo:.3f}–{hi:.3f})"

        ci_rows.append({
            "Metric": metric,
            "Point_Estimate": pt_val,
            "CI_Lower_95": lo,
            "CI_Upper_95": hi,
            "Formatted_Result": formatted,
            "Valid_Replicates": len(vals),
        })

    return pd.DataFrame(ci_rows), bootstrap_df


# ------------------------------------------------------------------------------
# Run bootstrap analysis (2000 replicates, 95% CI)
# ------------------------------------------------------------------------------
N_BOOTSTRAPS = 2000
CI_LEVEL = 95

print(f"Running stratified bootstrap ({N_BOOTSTRAPS:,} replicates; {CI_LEVEL}% CI)...")

test_metrics_ci_df, bootstrap_raw_df = bootstrap_metrics_ci(
    y_true=y_test,
    y_proba=y_test_proba,
    threshold=best_threshold,
    point_estimates=test_metrics,
    n_bootstraps=N_BOOTSTRAPS,
    ci=CI_LEVEL,
    random_state=SEED,
)

display_df = test_metrics_ci_df.copy()
display_df[["Point_Estimate", "CI_Lower_95", "CI_Upper_95"]] = display_df[["Point_Estimate", "CI_Lower_95", "CI_Upper_95"]].round(4)

print("=== Final Test-Set Metrics (95% Bootstrap CI) ===")
display(display_df)

# ------------------------------------------------------------------------------
# Save results and metadata
# ------------------------------------------------------------------------------
ci_path = TABLE_DIR / "test_metrics_bootstrap_95CI.csv"
raw_path = TABLE_DIR / "test_bootstrap_raw_metrics.csv"
meta_path = MODEL_INFO_DIR / "test_bootstrap_metadata.json"

test_metrics_ci_df.to_csv(ci_path, index=False)
bootstrap_raw_df.to_csv(raw_path, index=False)

bootstrap_metadata = {
    "dataset": "held-out_test_set",
    "resampling_method": "stratified_bootstrap_with_replacement",
    "n_bootstrap_replicates": N_BOOTSTRAPS,
    "confidence_interval_level": CI_LEVEL,
    "confidence_interval_method": "two-sided_percentile",
    "random_seed": int(SEED),
    "fixed_decision_threshold": float(best_threshold),
    "threshold_selection_source": "validation_set_maximum_F1",
    "n_test_observations": int(len(y_test)),
    "n_test_positive_outcomes": int(np.sum(np.asarray(y_test) == 1)),
    "n_test_negative_outcomes": int(np.sum(np.asarray(y_test) == 0)),
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(bootstrap_metadata, f, indent=4)

print(f"✓ Saved CI table to: {ci_path}")
print(f"✓ Saved raw replicates to: {raw_path}")
print(f"✓ Saved metadata to: {meta_path}")


In [ ]:
# ==============================================================================
# Cell 13: Export tables (Main & Supplementary)
# ==============================================================================

METRIC_LABELS = {
    "Accuracy": "Accuracy",
    "Precision_PPV": "Positive predictive value (PPV)",
    "Recall_Sensitivity": "Sensitivity",
    "Specificity": "Specificity",
    "NPV": "Negative predictive value (NPV)",
    "F1_score": "F1-score",
    "ROC_AUC": "Area under the ROC curve (AUROC)",
    "PR_AUC": "Area under the precision–recall curve (AUPRC)",
    "Brier": "Brier score",
    "Calibration_Slope": "Calibration slope",
    "Calibration_Intercept": "Calibration intercept",
}

detailed_table = test_metrics_ci_df[
    ["Metric", "Point_Estimate", "CI_Lower_95", "CI_Upper_95", "Formatted_Result"]
].copy()
detailed_table["Metric"] = detailed_table["Metric"].replace(METRIC_LABELS)
detailed_table = detailed_table.rename(
    columns={
        "Point_Estimate": "Point estimate",
        "CI_Lower_95": "95% CI lower",
        "CI_Upper_95": "95% CI upper",
        "Formatted_Result": "Estimate (95% CI)",
    }
)

manuscript_table = detailed_table[["Metric", "Estimate (95% CI)"]].copy()
print("=== Final Test-Set Performance ===")
display(manuscript_table)

manuscript_path = TABLE_DIR / "test_metrics_with_95CI_manuscript_table.csv"
detailed_path = TABLE_DIR / "test_metrics_with_95CI_detailed_table.csv"

manuscript_table.to_csv(manuscript_path, index=False)
detailed_table.to_csv(detailed_path, index=False)

print(f"✓ Manuscript table saved to: {manuscript_path}")
print(f"✓ Detailed table saved to: {detailed_path}")


In [ ]:
# ==============================================================================
# Cell 14: Central Figure Styling
# ==============================================================================

STYLE_PALETTE = {
    "primary": "#1f77b4",
    "secondary": "#ff7f0e",
    "neutral": "#7f7f7f",
    "danger": "#d62728",
    "success": "#2ca02c",
    "train": "#56B4E9",
    "val": "#E69F00",
    "test": "#009E73",
}


def set_publication_style():
    """Apply Matplotlib/Seaborn theme."""
    sns.set_theme(style="whitegrid", context="notebook", palette="deep")

    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.labelsize": 10,
        "axes.labelweight": "bold",
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "legend.title_fontsize": 9.5,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
        "axes.edgecolor": "#333333",
        "axes.linewidth": 0.9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#E5E5E5",
        "grid.linestyle": ":",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "lines.linewidth": 1.8,
        "lines.markersize": 5.5,
        "figure.dpi": 150,
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.05,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })
    
set_publication_style()
print("✓ style settings applied.")


In [ ]:
# ==============================================================================
# Cell 15: Figure Interpretation and Annotation Tools
# ==============================================================================

def _fmt(value, digits=3):
    """Safely format numeric values, using 'NA' for invalid values."""
    if value is None or not np.isfinite(value):
        return "NA"
    return f"{value:.{digits}f}"

def _fmt_pct(value, digits=1):
    """Safely format proportions as percentages, using 'NA' for invalid values."""
    if value is None or not np.isfinite(value):
        return "NA"
    return f"{100 * value:.{digits}f}%"

def save_interpretation(
    interpretation_text,
    interpretation_dict,
    filename_stem,
    output_dir=None,
):
    """Export figure interpretation text and structured data (JSON)."""
    if output_dir is None:
        output_dir = MODEL_INFO_DIR

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    txt_path = output_dir / f"{filename_stem}.txt"
    json_path = output_dir / f"{filename_stem}.json"

    txt_path.write_text(interpretation_text, encoding="utf-8")

    with json_path.open("w", encoding="utf-8") as f:
        json.dump(interpretation_dict, f, indent=4, ensure_ascii=False, default=str)

    print(f"✓ Interpretation saved: {txt_path.name}, {json_path.name}")
    return txt_path, json_path

def add_interpretation_box(ax, text, x=0.03, y=0.97, fontsize=8.2):
    ax.text(
        x,
        y,
        text,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=fontsize,
        wrap=True,
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": "white",
            "edgecolor": "#A0A0A0",
            "alpha": 0.92,
        },
    )

print("✓ Interpretation tools are ready.")


In [ ]:
# ==============================================================================
# Cell 16: Figure 1 — Confusion Matrices (Counts & Normalized) and interpretation
# ==============================================================================

y_test_pred = (y_test_proba >= best_threshold).astype(int)

cm_counts = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
cm_norm = confusion_matrix(y_test, y_test_pred, labels=[0, 1], normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
class_labels = ["No CLABSI (0)", "CLABSI (1)"]

# Panel A: Absolute counts
sns.heatmap(
    cm_counts, annot=True, fmt="d", cmap="Blues", cbar=False,
    xticklabels=class_labels, yticklabels=class_labels,
    linewidths=1, linecolor="#E5E5E5", ax=axes[0]
)
axes[0].set_title(f"Panel A: Absolute Counts (n = {len(y_test)})")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("True Label")

# Panel B: Normalized
sns.heatmap(
    cm_norm, annot=True, fmt=".1%", cmap="Blues", cbar=False,
    xticklabels=class_labels, yticklabels=class_labels,
    linewidths=1, linecolor="#E5E5E5", ax=axes[1]
)
axes[1].set_title(f"Panel B: Normalized (Threshold = {best_threshold:.2f})")
axes[1].set_xlabel("Predicted Label")

plt.suptitle("Figure 1: Test Set Confusion Matrices", y=1.02, fontsize=12, fontweight="bold")
plt.tight_layout()

for ext in ["png", "svg", "pdf"]:
    fig.savefig(FIG_DIR / f"figure_1_confusion_matrix_composite.{ext}")

plt.show()

tn, fp, fn, tp = cm_counts.ravel()
n_test = int(cm_counts.sum())

accuracy = (tp + tn) / n_test if n_test > 0 else np.nan
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan

if np.isfinite(sensitivity) and np.isfinite(specificity):
    if sensitivity >= 0.80 and specificity < 0.70:
        pattern = "High sensitivity with moderate-to-lower specificity."
    elif specificity >= 0.80 and sensitivity < 0.70:
        pattern = "High specificity with moderate-to-lower sensitivity."
    elif sensitivity >= 0.70 and specificity >= 0.70:
        pattern = "Balanced sensitivity and specificity (both ≥ 0.70)."
    else:
        pattern = "Suboptimal sensitivity and specificity (< 0.70)."
else:
    pattern = "Metrics undefined."

figure_1_text = f"""
FIGURE 1 — SUMMARY & INTERPRETATION
Threshold (Based on validation set): {_fmt(best_threshold, 3)}
Test sample size: {n_test}

Counts:
- True Negatives (TN): {tn} | False Positives (FP): {fp}
- False Negatives (FN): {fn} | True Positives (TP): {tp}

Metrics related to threshold:
- Accuracy: {_fmt_pct(accuracy)}
- Sensitivity (Recall): {_fmt_pct(sensitivity)}
- Specificity: {_fmt_pct(specificity)}
- PPV (Precision): {_fmt_pct(ppv)}
- NPV: {_fmt_pct(npv)}

Pattern:
{pattern}
""".strip()

save_interpretation(
    figure_1_text,
    {
        "figure": "Figure 1",
        "threshold": float(best_threshold),
        "n_test": n_test,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "accuracy": accuracy, "sensitivity": sensitivity, "specificity": specificity,
        "ppv": ppv, "npv": npv,
    },
    "figure_1_confusion_matrix_interpretation",
)

print(figure_1_text)


In [ ]:
# ==============================================================================
# Cell 17: Figure 2 — Discrimination Performance (ROC & PR Curves) & Interpretation
# ==============================================================================

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Panel A: ROC curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
test_roc_auc = roc_auc_score(y_test, y_test_proba)

axes[0].plot(fpr, tpr, color=STYLE_PALETTE["primary"], lw=2.2, label=f"Final SVM (AUROC = {test_roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color=STYLE_PALETTE["neutral"], lw=1.2, label="Chance Line (0.500)")
axes[0].set_xlim([-0.02, 1.02])
axes[0].set_ylim([-0.02, 1.02])
axes[0].set_xlabel("1 - Specificity (False Positive Rate)")
axes[0].set_ylabel("Sensitivity (True Positive Rate)")
axes[0].set_title("Panel A: ROC Curve")
axes[0].legend(loc="lower right", frameon=True)

# Panel B: Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_test_proba)
test_pr_auc = average_precision_score(y_test, y_test_proba)
prevalence = float(np.mean(y_test))

axes[1].plot(rec, prec, color=STYLE_PALETTE["primary"], lw=2.2, label=f"Final SVM (AUPRC = {test_pr_auc:.3f})")
axes[1].axhline(prevalence, linestyle="--", color=STYLE_PALETTE["danger"], lw=1.2, label=f"Prevalence Baseline ({prevalence:.1%})")
axes[1].set_xlim([-0.02, 1.02])
axes[1].set_ylim([-0.02, 1.02])
axes[1].set_xlabel("Recall (Sensitivity)")
axes[1].set_ylabel("Precision (PPV)")
axes[1].set_title("Panel B: PR Curve")
axes[1].legend(loc="upper right", frameon=True)

plt.suptitle("Figure 2: Held-out Test Set Discrimination", y=1.02, fontsize=12, fontweight="bold")
plt.tight_layout()

for ext in ["png", "svg", "pdf"]:
    fig.savefig(os.path.join(FIG_DIR, f"figure_2_discrimination_composite.{ext}"))

plt.show()
figure_2_text = f"""
FIGURE 2 — AUTOMATIC INTERPRETATION
Test-set performance:
- AUROC: {_fmt(test_roc_auc)}
- AUPRC: {_fmt(test_pr_auc)}
- Test-set CLABSI prevalence: {_fmt_pct(prevalence)}
""".strip()

save_interpretation(
    figure_2_text,
    {
        "figure": "Figure 2",
        "auroc": float(test_roc_auc),
        "auprc": float(test_pr_auc),
        "prevalence": float(prevalence),
    },
    "figure_2_discrimination_interpretation",
)

print(figure_2_text)


In [ ]:
# ==============================================================================
# Cell 18: Methodological Contract Configuration (Main Model & LOIO)
# ==============================================================================

CONTRACT_CONFIG = {
    "n_bins_calibration": 10,     
    "n_bins_histogram": 10,        
    "calibration_strategy": "quantile",
    "risk_percentiles": (20, 80),   
    "top_k_features": TOP_K_FEATURES,  
    "random_seed": 42,
    "main_cv_splits": 5,            
    "loio_group_col": "ICU Name",  
}


In [ ]:
# ==============================================================================
# Cell 19: Figure 3 — Calibration and Risk Distribution (Held-out Test Set)
# ==============================================================================

y_test_arr = np.asarray(y_test, dtype=int).ravel()
y_test_prob_arr = np.asarray(y_test_proba, dtype=float).ravel()

if y_test_arr.size != y_test_prob_arr.size:
    raise ValueError(
        f"y_test ({y_test_arr.size}) and y_test_proba ({y_test_prob_arr.size}) must have equal length."
    )

set_publication_style()
plt.close("all")

# Calibration summary: 10 quantile-based bins
n_cal_bins = CONTRACT_CONFIG["n_bins_calibration"]
quantile_bins = pd.qcut(
    pd.Series(y_test_prob_arr).rank(method="first"),
    q=n_cal_bins,
    labels=False,
    duplicates="drop",
)

cal_df = (
    pd.DataFrame(
        {"y_true": y_test_arr, "y_proba": y_test_prob_arr, "bin": quantile_bins.to_numpy()}
    )
    .groupby("bin", observed=False)
    .agg(
        mean_pred=("y_proba", "mean"),
        obs_rate=("y_true", "mean"),
        count=("y_true", "count"),
    )
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5))

# Panel A: Calibration curve
axes[0].plot(
    cal_df["mean_pred"],
    cal_df["obs_rate"],
    marker="o",
    lw=2,
    color=STYLE_PALETTE["primary"],
    label="Final SVM Pipeline",
)
axes[0].plot(
    [0, 1], [0, 1],
    linestyle="--",
    color=STYLE_PALETTE["neutral"],
    lw=1.2,
    label="Ideal Calibration",
)
axes[0].set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="Mean Predicted Probability",
    ylabel="Observed Event Rate",
    title=f"Panel A: Calibration Curve ({len(cal_df)} Quantile Bins)",
)
axes[0].legend(loc="lower right", frameon=True)

add_interpretation_box(
    axes[0],
    f"Held-out test set (n={len(y_test_arr)}).\nAssesses probability reliability.",
    x=0.03,
    y=0.97,
    fontsize=8,
)

# Panel B: Risk-score distribution
hist_bins = np.linspace(0, 1, CONTRACT_CONFIG["n_bins_histogram"] + 1)

for outcome, color, label in [
    (0, STYLE_PALETTE["primary"], "No CLABSI"),
    (1, STYLE_PALETTE["danger"], "CLABSI"),
]:
    axes[1].hist(
        y_test_prob_arr[y_test_arr == outcome],
        bins=hist_bins,
        density=True,
        alpha=0.6,
        color=color,
        edgecolor="white",
        linewidth=0.7,
        label=f"{label} (n = {np.sum(y_test_arr == outcome)})",
    )

axes[1].axvline(
    best_threshold,
    linestyle="--",
    color="black",
    lw=1.3,
    label=f"Validation F1 Cutoff ({best_threshold:.2f})",
)
axes[1].set(
    xlim=(0, 1),
    xlabel="Predicted Probability of CLABSI",
    ylabel="Density",
    title="Panel B: Predicted Risk Distribution",
)
axes[1].legend(loc="upper right", frameon=True)

add_interpretation_box(
    axes[1],
    "Outcome-stratified score distribution.\nAssesses class separation.",
    x=0.03,
    y=0.97,
    fontsize=8,
)

fig.suptitle(
    "Figure 3: Calibration and Predicted Risk Distribution",
    y=1.02,
    fontsize=12,
    fontweight="bold",
)
fig.tight_layout()

for ext in ["png", "svg", "pdf"]:
    fig.savefig(
        Path(FIG_DIR) / f"figure_3_calibration_composite.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight",
    )

display(fig)
plt.close(fig)

# Automatic interpretation
test_brier = brier_score_loss(y_test_arr, y_test_prob_arr)
cal_deviation = float(np.mean(np.abs(cal_df["mean_pred"] - cal_df["obs_rate"])))

mean_p_neg = (
    float(y_test_prob_arr[y_test_arr == 0].mean())
    if np.any(y_test_arr == 0) else np.nan
)
mean_p_pos = (
    float(y_test_prob_arr[y_test_arr == 1].mean())
    if np.any(y_test_arr == 1) else np.nan
)
risk_separation = (
    mean_p_pos - mean_p_neg
    if np.isfinite(mean_p_neg) and np.isfinite(mean_p_pos) else np.nan
)

fig3_text = f"""
FIGURE 3 — AUTOMATIC INTERPRETATION
Held-out test set (n = {len(y_test_arr)})

- Calibration bins: {len(cal_df)} quantile bins
- Brier score: {_fmt(test_brier)}
- Mean calibration deviation: {_fmt(cal_deviation)}
- Mean predicted risk — No CLABSI: {_fmt_pct(mean_p_neg)}
- Mean predicted risk — CLABSI: {_fmt_pct(mean_p_pos)}
- Mean risk separation: {_fmt_pct(risk_separation)}
- Validation-derived F1 threshold: {_fmt(best_threshold)}
This figure shows how well the predicted probabilities are calibrated and how the scores are distributed across the two outcome groups.
The threshold was selected using the validation set and is shown here on the test set for reference.


""".strip()

save_interpretation(
    fig3_text,
    {
        "n_samples": len(y_test_arr),
        "n_bins": len(cal_df),
        "brier_score": float(test_brier),
        "mean_cal_deviation": cal_deviation,
        "mean_risk_negative": mean_p_neg,
        "mean_risk_positive": mean_p_pos,
        "risk_separation": risk_separation,
        "threshold": float(best_threshold),
    },
    "figure_3_calibration_interpretation",
    output_dir=MODEL_INFO_DIR,
)

print(fig3_text)


In [ ]:
# ==============================================================================
# Cell 20: Split Performance Comparison & Automatic Interpretation
# ==============================================================================
# Compares final model performance across Train, Validation, and Test sets
# to evaluate internal generalization and assess potential overfitting.

set_publication_style()
plt.close("all")

# Prepare data for plotting
plot_metrics = [
    "Accuracy", "Precision_PPV", "Recall_Sensitivity", 
    "Specificity", "NPV", "F1_score", "ROC_AUC", "PR_AUC", "Brier"
]

metric_labels = {
    "Accuracy": "Accuracy", "Precision_PPV": "PPV", "Recall_Sensitivity": "Sensitivity",
    "Specificity": "Specificity", "NPV": "NPV", "F1_score": "F1-Score",
    "ROC_AUC": "AUROC", "PR_AUC": "AUPRC", "Brier": "Brier Score"
}

avail_metrics = [m for m in plot_metrics if m in metrics_comparison_df.columns]
df_plot = metrics_comparison_df.melt(
    id_vars="Dataset", value_vars=avail_metrics, var_name="Metric", value_name="Value"
)
df_plot["Metric_Label"] = df_plot["Metric"].map(metric_labels)

dataset_order = [d for d in ["Train", "Validation", "Test"] if d in df_plot["Dataset"].unique()]

# Render comparison plot
fig, ax = plt.subplots(figsize=(12, 5.5))

sns.barplot(
    data=df_plot,
    x="Metric_Label",
    y="Value",
    hue="Dataset",
    hue_order=dataset_order,
    palette={
        "Train": STYLE_PALETTE["train"],
        "Validation": STYLE_PALETTE["val"],
        "Test": STYLE_PALETTE["test"],
    },
    edgecolor="white",
    linewidth=0.7,
    ax=ax,
)

ax.set(ylim=(0, 1.15), xlabel="Evaluation Metric", ylabel="Metric Value")
ax.set_title("Main Model Performance Across Development and Held-Out Test sets", pad=12)
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Dataset Split", loc="upper right", frameon=True)

add_interpretation_box(
    ax,
    "Development vs. Held-out test performance.\nHigher values show better performance (except Brier score).",
    x=0.02, y=0.98, fontsize=8.0
)

fig.tight_layout()

# Save figure in publication formats
fig_dir_path = Path(FIG_DIR)
fig_dir_path.mkdir(parents=True, exist_ok=True)
for ext in ["png", "pdf", "svg"]:
    fig.savefig(
        fig_dir_path / f"main_split_performance_comparison.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight"
    )

plt.show()
plt.close(fig)

# Automated diagnostic evaluation and train-test gap analysis
metric_df = metrics_comparison_df.set_index("Dataset")

train_auroc = float(metric_df.loc["Train", "ROC_AUC"])
val_auroc = float(metric_df.loc["Validation", "ROC_AUC"])
test_auroc = float(metric_df.loc["Test", "ROC_AUC"])

train_pr = float(metric_df.loc["Train", "PR_AUC"])
val_pr = float(metric_df.loc["Validation", "PR_AUC"])
test_pr = float(metric_df.loc["Test", "PR_AUC"])

train_brier = float(metric_df.loc["Train", "Brier"])
val_brier = float(metric_df.loc["Validation", "Brier"])
test_brier = float(metric_df.loc["Test", "Brier"])

auroc_gap = train_auroc - test_auroc

if np.isfinite(auroc_gap):
    if abs(auroc_gap) < 0.05:
        gap_comment = "The Train–Test AUROC difference is small (<0.05), indicating minimal overfitting."
    elif auroc_gap > 0:
        gap_comment = "Test AUROC is lower than Train AUROC, reflecting expected generalization gap."
    else:
        gap_comment = "Test AUROC slightly is bigger than Train AUROC, attributable to sample variability."
else:
    gap_comment = "Train–Test difference could not be evaluated."

perf_text = f"""
MAIN SPLIT PERFORMANCE — AUTOMATIC INTERPRETATION

AUROC:
- Train: {_fmt(train_auroc)} | Validation: {_fmt(val_auroc)} | Test: {_fmt(test_auroc)}
- Train–Test Gap: {_fmt(auroc_gap)}

AUPRC:
- Train: {_fmt(train_pr)} | Validation: {_fmt(val_pr)} | Test: {_fmt(test_pr)}

Brier Score:
- Train: {_fmt(train_brier)} | Validation: {_fmt(val_brier)} | Test: {_fmt(test_brier)}

Interpretation:
{gap_comment}
""".strip()

save_interpretation(
    interpretation_text=perf_text,
    interpretation_dict={
        "train_auroc": train_auroc, "val_auroc": val_auroc, "test_auroc": test_auroc,
        "train_pr_auc": train_pr, "val_pr_auc": val_pr, "test_pr_auc": test_pr,
        "train_brier": train_brier, "val_brier": val_brier, "test_brier": test_brier,
        "auroc_gap": auroc_gap, "interpretation": gap_comment
    },
    filename_stem="main_split_performance_interpretation",
    output_dir=MODEL_INFO_DIR
)

print(perf_text)


In [ ]:
# ==============================================================================
# Cell 21: Learning Curve & Simple Interpretation
# ==============================================================================
# Shows how the model improves as it sees more training data.
# Helps check if the model is overfitting or needs more data.

# Prepare data and compute learning curve
selected_feature_names = list(top_15_features)
X_train_lc = X_train.loc[:, selected_feature_names].copy()
y_train_array = np.asarray(y_train).ravel()

learning_cv = StratifiedKFold(
    n_splits=CONTRACT_CONFIG.get("main_cv_splits", 5),
    shuffle=True,
    random_state=SEED
)
learning_curve_estimator = clone(final_pipeline)

train_sizes, train_scores, cv_scores = learning_curve(
    estimator=learning_curve_estimator,
    X=X_train_lc,
    y=y_train_array,
    cv=learning_cv,
    scoring="roc_auc",
    train_sizes=np.linspace(0.2, 1.0, 5),
    shuffle=True,
    random_state=SEED,
    n_jobs=-1,
    return_times=False,
)

# Calculate average scores and 95% confidence intervals
n_folds = learning_cv.get_n_splits()
t_critical = stats.t.ppf(0.975, df=n_folds - 1)

train_scores_mean = np.mean(train_scores, axis=1)
cv_scores_mean = np.mean(cv_scores, axis=1)

train_scores_sd = np.std(train_scores, axis=1, ddof=1)
cv_scores_sd = np.std(cv_scores, axis=1, ddof=1)

train_ci95_margin = t_critical * train_scores_sd / np.sqrt(n_folds)
cv_ci95_margin = t_critical * cv_scores_sd / np.sqrt(n_folds)

train_ci95_lower = np.clip(train_scores_mean - train_ci95_margin, 0.0, 1.0)
train_ci95_upper = np.clip(train_scores_mean + train_ci95_margin, 0.0, 1.0)
cv_ci95_lower = np.clip(cv_scores_mean - cv_ci95_margin, 0.0, 1.0)
cv_ci95_upper = np.clip(cv_scores_mean + cv_ci95_margin, 0.0, 1.0)

# Plot the learning curve
fig, ax = plt.subplots(figsize=(8, 5.5))

# Training score line
ax.plot(
    train_sizes, train_scores_mean,
    marker="o", lw=2, color=STYLE_PALETTE["primary"], label="Training AUROC"
)
ax.fill_between(
    train_sizes, train_ci95_lower, train_ci95_upper,
    color=STYLE_PALETTE["primary"], alpha=0.18, label="Training 95% CI"
)

# Cross-validation score line
ax.plot(
    train_sizes, cv_scores_mean,
    marker="s", lw=2, color=STYLE_PALETTE["secondary"], label="Cross-Validation AUROC"
)
ax.fill_between(
    train_sizes, cv_ci95_lower, cv_ci95_upper,
    color=STYLE_PALETTE["secondary"], alpha=0.18, label="CV 95% CI"
)

ax.set(
    ylim=(0.45, 1.02),
    xlabel="Number of Training Patients",
    ylabel="AUROC Score",
    title="Learning Curve: Model Performance vs. Training Size"
)
ax.grid(True, linestyle=":", linewidth=0.8, alpha=0.65)
ax.legend(loc="lower right", frameon=True)

add_interpretation_box(
    ax,
    "As training data grows, the gap between curves narrows,\nshowing that the model learns stably.",
    x=0.03, y=0.97, fontsize=8.0
)

plt.tight_layout()

# Save the plot in all formats
for ext in ["png", "svg", "pdf"]:
    fig.savefig(
        Path(FIG_DIR) / f"learning_curve.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight"
    )

plt.show()
plt.close(fig)

# Summary and simple automatic interpretation
final_train_auroc = float(train_scores_mean[-1])
final_cv_auroc = float(cv_scores_mean[-1])
final_gap = final_train_auroc - final_cv_auroc

if final_gap < 0.10:
    gap_diag = "The difference between training and validation is small, meaning the model generalizes well."
else:
    gap_diag = "There is a noticeable gap between training and validation, which may suggest mild overfitting."

learning_text = f"""
LEARNING CURVE — SUMMARY

- Final Training AUROC: {_fmt(final_train_auroc)}
- Final Validation AUROC: {_fmt(final_cv_auroc)}
- Difference (Gap): {_fmt(final_gap)}

Conclusion:
{gap_diag}
""".strip()

save_interpretation(
    learning_text,
    {
        "final_train_auroc": final_train_auroc,
        "final_cv_auroc": final_cv_auroc,
        "final_gap": final_gap,
        "interpretation": gap_diag
    },
    "learning_curve_interpretation",
    output_dir=MODEL_INFO_DIR
)

print(learning_text)


In [ ]:
# ==============================================================================
# Cell 22: Decision Curve Analysis (DCA) & Clinical Utility Interpretation
# ==============================================================================
# Evaluates clinical usefulness by comparing the model's net benefit against
# two default strategies: "Treat All" (intervene on everyone) and "Treat None".

# Calculate net benefit across decision thresholds
def calculate_decision_curve(y_true, y_proba, thresholds=None):
    y_true = np.asarray(y_true, dtype=int).ravel()
    y_proba = np.asarray(y_proba, dtype=float).ravel()

    if thresholds is None:
        thresholds = np.arange(0.01, 0.99, 0.01)

    n = len(y_true)
    prevalence = float(np.mean(y_true))
    rows = []

    for pt in thresholds:
        y_pred = (y_proba >= pt).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        weight = pt / (1.0 - pt)
        
        # Net benefit formulas
        net_benefit_model = (tp / n) - (fp / n) * weight
        net_benefit_all = prevalence - (1.0 - prevalence) * weight
        
        rows.append({
            "Threshold": pt,
            "Net_Benefit_Model": net_benefit_model,
            "Net_Benefit_Treat_All": net_benefit_all,
            "Net_Benefit_Treat_None": 0.0,
        })

    return pd.DataFrame(rows)

dca_df = calculate_decision_curve(y_test, y_test_proba)

# Plot the decision curve
fig, ax = plt.subplots(figsize=(7.5, 5.5))

ax.plot(
    dca_df["Threshold"], dca_df["Net_Benefit_Model"],
    color=STYLE_PALETTE["primary"], lw=2.2, label="Final SVM Model"
)
ax.plot(
    dca_df["Threshold"], dca_df["Net_Benefit_Treat_All"],
    linestyle="--", color=STYLE_PALETTE["danger"], lw=1.6, label="Treat All"
)
ax.plot(
    dca_df["Threshold"], dca_df["Net_Benefit_Treat_None"],
    linestyle=":", color=STYLE_PALETTE["neutral"], lw=1.8, label="Treat None"
)
ax.axvline(
    best_threshold,
    linestyle="-.", color="black", lw=1.2,
    label=f"Selected Cutoff ({best_threshold:.2f})"
)

ax.set(
    xlim=(0.0, 1.0),
    ylim=(-0.05, max(0.2, float(dca_df["Net_Benefit_Model"].max()) * 1.15)),
    xlabel="Threshold Probability (Pt)",
    ylabel="Net Benefit",
    title="Decision Curve Analysis: Clinical Net Benefit"
)
ax.legend(loc="upper right", frameon=True)

add_interpretation_box(
    ax,
    "The model provides clinical value when its curve\nsits above both 'Treat All' and 'Treat None'.",
    x=0.03, y=0.97, fontsize=8.0
)

plt.tight_layout()

for ext in ["png", "svg", "pdf"]:
    fig.savefig(
        Path(FIG_DIR) / f"decision_curve_analysis.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight"
    )

plt.show()
plt.close(fig)

# Automatic clinical interpretation
dca_thresholds = np.asarray(dca_df["Threshold"], dtype=float)
dca_model = np.asarray(dca_df["Net_Benefit_Model"], dtype=float)
dca_all = np.asarray(dca_df["Net_Benefit_Treat_All"], dtype=float)

# Evaluate over typical clinical range (5% to 50% risk thresholds)
clinical_mask = (dca_thresholds >= 0.05) & (dca_thresholds <= 0.50)
model_mean_nb = float(np.nanmean(dca_model[clinical_mask]))
all_mean_nb = float(np.nanmean(dca_all[clinical_mask]))
better_than_all = float(np.nanmean(dca_model[clinical_mask] > dca_all[clinical_mask]))

# Performance at the chosen threshold
threshold_idx = int(np.argmin(np.abs(dca_thresholds - float(best_threshold))))
nb_at_cutoff = float(dca_model[threshold_idx])
all_nb_at_cutoff = float(dca_all[threshold_idx])

if nb_at_cutoff > all_nb_at_cutoff and nb_at_cutoff > 0:
    clinical_summary = "Using the model at this cutoff gives higher net benefit than treating everyone or no one."
else:
    clinical_summary = "At this cutoff, the model offers limited additional net benefit compared to default strategies."

dca_text = f"""
DECISION CURVE ANALYSIS — SUMMARY

Operating cutoff based on validation: {_fmt(best_threshold)}

At selected cutoff:
- Model net benefit: {_fmt(nb_at_cutoff)}
- Treat-all net benefit: {_fmt(all_nb_at_cutoff)}

Across standard clinical range (5%–50% risk):
- Average model net benefit: {_fmt(model_mean_nb)}
- Average treat-all net benefit: {_fmt(all_mean_nb)}
- Model beats Treat-All in: {_fmt_pct(better_than_all)} of thresholds

Conclusion:
{clinical_summary}
""".strip()

save_interpretation(
    dca_text,
    {
        "operating_threshold": float(best_threshold),
        "nb_at_operating_threshold": nb_at_cutoff,
        "mean_model_nb": model_mean_nb,
        "better_than_all_pct": better_than_all,
    },
    "decision_curve_analysis_interpretation",
    output_dir=MODEL_INFO_DIR,
)

print(dca_text)


In [ ]:
# ==============================================================================
# Cell 23: SHAP Feature Importance & Automatic Interpretation
# ==============================================================================
# Explains model predictions on the held-out test set using SHAP values.
# Shows which clinical features have the strongest impact and their direction.

#Transform features using the fitted preprocessor
fitted_preprocessor = final_pipeline.named_steps["preprocessor"]
svm_step = final_pipeline.named_steps["classifier"]

try:
    transformed_feature_names = list(fitted_preprocessor.get_feature_names_out())
except Exception:
    transformed_feature_names = [f"feature_{i}" for i in range(len(top_15_features))]

X_train_proc = fitted_preprocessor.transform(X_train[top_15_features])
X_test_proc = fitted_preprocessor.transform(X_test[top_15_features])

X_train_proc_df = pd.DataFrame(X_train_proc, columns=transformed_feature_names)
X_test_proc_df = pd.DataFrame(X_test_proc, columns=transformed_feature_names)

# Compute SHAP values with LinearExplainer
background = shap.sample(X_train_proc_df, min(100, len(X_train_proc_df)), random_state=SEED)
explainer = shap.LinearExplainer(svm_step, background)
shap_explanation = explainer(X_test_proc_df)
shap_array = shap_explanation.values if hasattr(shap_explanation, "values") else shap_explanation

# Render and save the beeswarm summary plot
fig = plt.figure(figsize=(9, 6.5))
shap.summary_plot(shap_array, X_test_proc_df, max_display=15, show=False)
plt.title("SHAP Feature Importance (Test Set)", fontsize=13, pad=12)
plt.tight_layout()
fig_dir_path = Path(FIG_DIR)
fig_dir_path.mkdir(parents=True, exist_ok=True)
for ext in ["png", "svg", "pdf"]:
    fig.savefig(
        fig_dir_path / f"final_model_shap_summary_beeswarm.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight",
    )

plt.show()
plt.close(fig)

# Rank features and generate simple automatic interpretation
shap_importance_df = pd.DataFrame({
    "Feature": transformed_feature_names,
    "Mean_Absolute_SHAP": np.abs(shap_array).mean(axis=0),
    "Mean_SHAP": shap_array.mean(axis=0),
}).sort_values("Mean_Absolute_SHAP", ascending=False).reset_index(drop=True)

top_5_features = shap_importance_df.head(5)["Feature"].tolist()
feature_list_str = "\n".join([f"  {i+1}. {feat}" for i, feat in enumerate(top_5_features)])

shap_text = f"""
SHAP FEATURE IMPORTANCE — SUMMARY
Evaluated on held-out Test Set (n = {len(X_test_proc_df)})

Top 5 most impactful features:
{feature_list_str}

Key Takeaway:
- Red dots indicate high feature values; blue dots indicate low feature values.
- Points to the right push risk predictions higher; points to the left lower the risk.
- Note: SHAP values show statistical associations with the model, not causal clinical effects.
""".strip()

save_interpretation(
    shap_text,
    {
        "top_15_features": shap_importance_df.head(15)["Feature"].tolist(),
        "mean_abs_shap": shap_importance_df.head(15)["Mean_Absolute_SHAP"].tolist(),
    },
    "figure_4_shap_interpretation",
    output_dir=MODEL_INFO_DIR,
)

print(shap_text)


In [ ]:
# ==============================================================================
# Cell 24: Linear SVM Coefficients & Interpretation
# ==============================================================================
# Lists the weights (coefficients) assigned by the model to each feature.
# These show how much each feature affects the final prediction.

# Extract model coefficients and rank them
coef = svm_step.coef_.ravel()

coef_df = pd.DataFrame({
    "Feature": transformed_feature_names,
    "Coefficient": coef,
    "Absolute_Coefficient": np.abs(coef),
}).sort_values("Absolute_Coefficient", ascending=False).reset_index(drop=True)

# Display the top 20 features
print("Top 20 Model Coefficients:")
display(coef_df.head(20))

# Save the full table to a CSV file
table_dir_path = Path(TABLE_DIR)
table_dir_path.mkdir(parents=True, exist_ok=True)
coef_df.to_csv(table_dir_path / "linear_svm_coefficients.csv", index=False)

# Automated interpretation
top_feature = coef_df.iloc[0]["Feature"]
top_weight = coef_df.iloc[0]["Coefficient"]

coef_text = f"""
LINEAR SVM COEFFICIENTS — SUMMARY

The strongest weight in the model belongs to '{top_feature}' (Weight: {_fmt(top_weight)}).

Key Points:
- The sign (positive/negative) shows the direction of the relationship.
- The size (magnitude) shows how strongly the model relies on that feature.
- Important: These weights are calculated on transformed features (e.g., after scaling)
  and are not simple clinical odds ratios.
""".strip()

save_interpretation(
    coef_text,
    {"top_features_by_abs_coef": coef_df.head(15)["Feature"].tolist()},
    "linear_svm_coefficients_interpretation",
    output_dir=MODEL_INFO_DIR
)

print(coef_text)


In [ ]:
# ==============================================================================
# Cell 25: Leakage-Free Risk Stratification & Test Set Evaluation
# ==============================================================================
# Splits patients into Low, Intermediate, and High risk groups.
# Cutoffs are calculated on the Validation set (20th and 80th percentiles)
# and then applied to the Test set to prevent any data leakage.

# Prepare and check probability arrays
y_val_proba_arr = np.asarray(y_val_proba, dtype=float).ravel()
y_test_proba_arr = np.asarray(y_test_proba, dtype=float).ravel()
y_test_arr = np.asarray(y_test, dtype=int).ravel()

if y_test_arr.size != y_test_proba_arr.size:
    raise ValueError("Test labels and test probabilities must have the same size.")

# Use risk cutoffs strictly from the validation set
LOW_CUT_PERCENTILE, HIGH_CUT_PERCENTILE = CONTRACT_CONFIG.get("risk_percentiles", (20, 80))
LOW_CUT = float(np.percentile(y_val_proba_arr, LOW_CUT_PERCENTILE))
HIGH_CUT = float(np.percentile(y_val_proba_arr, HIGH_CUT_PERCENTILE))

RISK_LABELS = ["Low Risk", "Intermediate Risk", "High Risk"]
RISK_COLORS = [
    STYLE_PALETTE["primary"],
    STYLE_PALETTE["secondary"],
    STYLE_PALETTE["danger"],
]

print(f"Risk cutoffs calculated from Validation set:")
print(f"  - Low Risk Cutoff (<{LOW_CUT_PERCENTILE}th percentile):  {LOW_CUT:.4f}")
print(f"  - High Risk Cutoff (>{HIGH_CUT_PERCENTILE}th percentile): {HIGH_CUT:.4f}")

cutoff_metadata = {
    "low_cut": LOW_CUT,
    "high_cut": HIGH_CUT,
    "low_cut_percentile": LOW_CUT_PERCENTILE,
    "high_cut_percentile": HIGH_CUT_PERCENTILE,
    "source": "Validation set only (leakage-free)",
}
with open(Path(MODEL_INFO_DIR) / "risk_cutoffs.json", "w", encoding="utf-8") as f:
    json.dump(cutoff_metadata, f, indent=4)

# Helper functions for risk grouping and summary table
def assign_risk_groups(probabilities, low_cut=LOW_CUT, high_cut=HIGH_CUT):
    """Assigns each predicted risk to Low, Intermediate, or High risk group."""
    probs = np.asarray(probabilities, dtype=float).ravel()
    groups = np.select(
        [probs <= low_cut, probs <= high_cut],
        ["Low Risk", "Intermediate Risk"],
        default="High Risk",
    )
    return pd.Categorical(groups, categories=RISK_LABELS, ordered=True)

def build_risk_table(y_true, probabilities, low_cut=LOW_CUT, high_cut=HIGH_CUT):
    """Builds a detailed summary table of CLABSI rates across risk groups."""
    risk_groups = assign_risk_groups(probabilities, low_cut=low_cut, high_cut=high_cut)
    df = pd.DataFrame({"y_true": y_true, "y_proba": probabilities, "Risk_Group": risk_groups})

    low_group = df[df["Risk_Group"] == "Low Risk"]
    low_events = int(low_group["y_true"].sum())
    low_total = len(low_group)
    low_rate = low_events / low_total if low_total > 0 else np.nan

    rows = []
    for group in RISK_LABELS:
        grp = df[df["Risk_Group"] == group]
        n_patients = len(grp)
        n_events = int(grp["y_true"].sum())
        obs_rate = n_events / n_patients if n_patients > 0 else np.nan

        # 95% Wilson Confidence Interval
        if n_patients > 0:
            ci_low, ci_high = proportion_confint(n_events, n_patients, alpha=0.05, method="wilson")
            ci_str = f"{ci_low*100:.1f}%–{ci_high*100:.1f}%"
        else:
            ci_str = "N/A"

        # Relative Risk & Fisher's Exact Test vs Low Risk group
        if group == "Low Risk":
            rr, p_val = 1.0, np.nan
        elif n_patients > 0 and low_total > 0 and pd.notna(low_rate) and low_rate > 0:
            rr = obs_rate / low_rate
            _, p_val = fisher_exact([[n_events, n_patients - n_events], [low_events, low_total - low_events]])
        else:
            rr, p_val = np.nan, np.nan

        p_str = "<0.001" if (pd.notna(p_val) and p_val < 0.001) else (f"{p_val:.4f}" if pd.notna(p_val) else "N/A")

        rows.append({
            "Risk Group": group,
            "Total Patients": n_patients,
            "Infection Events": n_events,
            "Observed Rate (%)": round(obs_rate * 100, 1) if pd.notna(obs_rate) else np.nan,
            "95% CI Rate": ci_str,
            "Relative Risk vs Low": round(rr, 2) if pd.notna(rr) else np.nan,
            "P-Value vs Low": p_str,
            "Average Predicted Risk (%)": round(grp["y_proba"].mean() * 100, 1) if n_patients > 0 else np.nan,
        })

    return pd.DataFrame(rows), df

# Generate table on the held-out test set
risk_table, test_risk_df = build_risk_table(y_test_arr, y_test_proba_arr, LOW_CUT, HIGH_CUT)
risk_table_path = Path(TABLE_DIR) / f"risk_stratification_table_p{LOW_CUT_PERCENTILE}_p{HIGH_CUT_PERCENTILE}.csv"
risk_table.to_csv(risk_table_path, index=False, encoding="utf-8-sig")

print("\n--- Test Set Risk Stratification Table ---")
display(risk_table)
print(f"Table saved to: {risk_table_path}")

# Simple automatic interpretation
low_rate = risk_table.loc[risk_table["Risk Group"] == "Low Risk", "Observed Rate (%)"].values[0]
high_rate = risk_table.loc[risk_table["Risk Group"] == "High Risk", "Observed Rate (%)"].values[0]
high_rr = risk_table.loc[risk_table["Risk Group"] == "High Risk", "Relative Risk vs Low"].values[0]

risk_text = f"""
RISK STRATIFICATION — SUMMARY (HELD-OUT TEST SET)

- Low Risk Group: {low_rate}% observed infection rate.
- High Risk Group: {high_rate}% observed infection rate.
- High vs. Low Relative Risk: {high_rr}x higher risk.

Conclusion:
The model clearly separates patients into distinct clinical risk levels.
Patients in the High Risk tier experienced significantly more CLABSI events than those in the Low Risk group.
""".strip()

save_interpretation(
    interpretation_text=risk_text,
    interpretation_dict={
        "low_risk_observed_rate": low_rate,
        "high_risk_observed_rate": high_rate,
        "high_risk_relative_risk": high_rr,
        "low_cut": LOW_CUT,
        "high_cut": HIGH_CUT,
    },
    filename_stem="risk_stratification_test_interpretation",
    output_dir=MODEL_INFO_DIR,
)

print("\n" + risk_text)


In [ ]:
# ==============================================================================
# Cell 26: Risk Stratification Plot & Automatic Interpretation
# ==============================================================================
# Plots observed CLABSI rates across Low, Intermediate, and High risk groups
# with 95% Wilson confidence intervals on the held-out test set.

# Prepare data for plotting
risk_plot_rows = []
for group in RISK_LABELS:
    grp = test_risk_df[test_risk_df["Risk_Group"] == group]
    n_pts = len(grp)
    n_events = int(grp["y_true"].sum())

    if n_pts > 0:
        event_rate = n_events / n_pts
        ci_low, ci_high = proportion_confint(
            count=n_events, nobs=n_pts, alpha=0.05, method="wilson"
        )
    else:
        event_rate, ci_low, ci_high = np.nan, np.nan, np.nan

    risk_plot_rows.append({
        "Risk_Group": group,
        "N": n_pts,
        "Event_Rate_%": 100 * event_rate if pd.notna(event_rate) else 0.0,
        "CI_Low_%": 100 * ci_low if pd.notna(ci_low) else 0.0,
        "CI_High_%": 100 * ci_high if pd.notna(ci_high) else 0.0,
    })

risk_plot_df = pd.DataFrame(risk_plot_rows)

# Render bar chart
fig, ax = plt.subplots(figsize=(7.5, 5.5))

x_pos = np.arange(len(risk_plot_df))
rates = risk_plot_df["Event_Rate_%"].to_numpy(dtype=float)
ci_lows = risk_plot_df["CI_Low_%"].to_numpy(dtype=float)
ci_highs = risk_plot_df["CI_High_%"].to_numpy(dtype=float)

yerr_lower = np.clip(rates - ci_lows, 0.0, None)
yerr_upper = np.clip(ci_highs - rates, 0.0, None)

bars = ax.bar(
    x_pos,
    rates,
    yerr=np.vstack([yerr_lower, yerr_upper]),
    capsize=5,
    width=0.65,
    color=RISK_COLORS,
    edgecolor="#333333",
    linewidth=0.8,
    error_kw={"elinewidth": 1.0, "ecolor": "#333333"},
)

# Set clean category labels with probability ranges
x_labels = [
    f"Low Risk\n(≤ {100 * LOW_CUT:.1f}%)",
    f"Intermediate Risk\n({100 * LOW_CUT:.1f}%–{100 * HIGH_CUT:.1f}%)",
    f"High Risk\n(> {100 * HIGH_CUT:.1f}%)",
]
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels)

ax.set(
    ylabel="Observed CLABSI Rate (%)",
    xlabel="Risk Group (Cutoffs derived from validation)",
    title="Observed Infection Rates by Risk group (Held-Out Test Set)",
    ylim=(0, max(1.0, float(np.max(rates)) * 1.30)),
)

for index, bar in enumerate(bars):
    n_val = int(risk_plot_df.loc[index, "N"])
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.03 * ax.get_ylim()[1],
        f"n = {n_val}",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold",
    )

ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.65)
ax.grid(axis="x", visible=False)

add_interpretation_box(
    ax,
    "Bars show observed event rates; error bars show 95% Wilson CIs.\nRisk cutoffs were derived from validation probabilities.",
    x=0.03, y=0.97, fontsize=8.0
)

plt.tight_layout()

figure_stem = f"risk_stratification_validation_p{LOW_CUT_PERCENTILE}_p{HIGH_CUT_PERCENTILE}"
fig_dir_path = Path(FIG_DIR)
fig_dir_path.mkdir(parents=True, exist_ok=True)

for ext in ["png", "svg", "pdf"]:
    fig.savefig(
        fig_dir_path / f"{figure_stem}.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight",
    )

plt.show()
plt.close(fig)

#Automatic interpretation
summary_lines = [
    f"- {row['Risk_Group']}: {row['Event_Rate_%']:.1f}% rate (95% CI: {row['CI_Low_%']:.1f}%–{row['CI_High_%']:.1f}%, n = {row['N']})"
    for _, row in risk_plot_df.iterrows()
]

plot_text = f"""
RISK STRATIFICATION PLOT — SUMMARY

Cutoffs (frozen from validation percentiles):
- Low cut (<{LOW_CUT_PERCENTILE}th percentile): {100 * LOW_CUT:.1f}%
- High cut (>{HIGH_CUT_PERCENTILE}th percentile): {100 * HIGH_CUT:.1f}%

Observed CLABSI rates on held-out test data:
{chr(10).join(summary_lines)}

Key Takeaway:
The visual gradient shows a clear separation between the three risk groups.
""".strip()

save_interpretation(
    interpretation_text=plot_text,
    interpretation_dict={
        "figure": figure_stem,
        "low_cut": LOW_CUT,
        "high_cut": HIGH_CUT,
        "risk_group_summary": risk_plot_df.to_dict(orient="records"),
    },
    filename_stem=f"{figure_stem}_interpretation",
    output_dir=MODEL_INFO_DIR,
)

print(plot_text)


In [ ]:
# ==============================================================================
# Cell 27: LOIO Setup
# ==============================================================================
# Sets up Leave-One-ICU-Out validation

if "SEED" not in globals():
    SEED = CONTRACT_CONFIG.get("random_seed", 42)

np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

if "set_publication_style" in globals():
    set_publication_style()

# Check data and required columns
group_column = CONTRACT_CONFIG.get("loio_group_col", "ICU Name")
target = "CLABSI" if "target" not in globals() else target

if "df" not in globals() and "data" in globals():
    df = data.copy()

if "df" not in globals():
    raise RuntimeError("Dataset DataFrame ('df' or 'data') was not found.")

if group_column not in df.columns:
    raise ValueError(f"ICU column '{group_column}' was not found in the data.")

if target not in df.columns:
    raise ValueError(f"Target column '{target}' was not found in the data.")

all_features = [col for col in df.columns if col not in [target, group_column]]

# Check shared model utilities
required_shared = [
    "categorical_features",
    "numerical_features",
    "TOP_K_FEATURES",
    "param_grid",
    "select_features_and_tune",
    "tune_threshold_on_val",
]

missing_shared = [item for item in required_shared if item not in globals()]
if missing_shared:
    raise RuntimeError(f"Missing shared components: {', '.join(missing_shared)}")

# LOIO settings and output folders
LOIO_SCORING = "roc_auc"
LOIO_MAX_INNER_SPLITS = CONTRACT_CONFIG.get("main_cv_splits", 5)

LOIO_OUTPUT_DIR = (
    Path(OUTPUT_DIR) / "loio"
    if "OUTPUT_DIR" in globals()
    else Path("results/loio")
)
LOIO_FIG_DIR = LOIO_OUTPUT_DIR / "figures"
LOIO_TABLE_DIR = LOIO_OUTPUT_DIR / "tables"
LOIO_MODEL_INFO_DIR = LOIO_OUTPUT_DIR / "model_info"

for folder in [LOIO_OUTPUT_DIR, LOIO_FIG_DIR, LOIO_TABLE_DIR, LOIO_MODEL_INFO_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("=== LOIO Setup Complete ===")
print("Validation design:     Leave-One-ICU-Out within Shiraz Namazi hospital ICUs")
print(f"Target variable:       {target}")
print(f"ICU column:            {group_column}")
print(f"Available features:    {len(all_features)}")
print(f"Top-K features:        {TOP_K_FEATURES} (same as main model)")
print(f"Output directory:      {LOIO_OUTPUT_DIR.resolve()}")


In [ ]:
# ==============================================================================
# Cell 28: Leakage-Free LOIO Cross-Validation
# ==============================================================================
# Tests the model in each left-out ICU .
# Feature selection, calibration, and threshold selection use development data only.

unique_icus = sorted(df[group_column].dropna().unique().tolist())
n_icus = len(unique_icus)

print(f"Starting LOIO cross-validation across {n_icus} ICUs...")

loio_fold_results = []
loio_patient_predictions = []
fold_selected_features = {}

for fold_idx, test_icu in enumerate(unique_icus, start=1):
    print(f"\n[{fold_idx}/{n_icus}] Testing left-out ICU: '{test_icu}'")

    # Keep one ICU unseen for testing
    df_dev = df[df[group_column] != test_icu].copy()
    df_test = df[df[group_column] == test_icu].copy()

    y_dev_full = df_dev[target].astype(int).to_numpy()
    y_test_icu = df_test[target].astype(int).to_numpy()

    test_n = len(df_test)
    test_events = int(y_test_icu.sum())

    # Use development data only for training, calibration, and threshold selection
    df_dev_train, df_dev_val, y_dev_train, y_dev_val = train_test_split(
        df_dev,
        y_dev_full,
        test_size=0.25,
        stratify=y_dev_full,
        random_state=SEED,
    )

    # Select features and tune the model using development training data
    fold_result = select_features_and_tune(
        df_dev_train,
        y_dev_train,
        categorical_features,
        numerical_features,
        param_grid,
        seed=SEED,
        top_k=TOP_K_FEATURES,
        cv_splits=LOIO_MAX_INNER_SPLITS,
        scoring=LOIO_SCORING,
    )

    fold_pipeline = fold_result["final_pipeline"]
    selected_cols = fold_result["selected_features"]
    fold_selected_features[test_icu] = selected_cols

    # Calibrate probabilities and select the threshold on development validation data
    fold_calibrator, best_threshold_fold, _ = calibrate_and_tune_threshold(
        fold_pipeline,
        df_dev_val,
        y_dev_val,
        selected_cols,
        method=CALIBRATION_METHOD,
    )

    # Test once on the left-out ICU
    test_probs = apply_calibrator(
        fold_pipeline,
        fold_calibrator,
        df_test,
        selected_cols,
        method=CALIBRATION_METHOD,
    )
    test_preds = (test_probs >= best_threshold_fold).astype(int)

    # Save patient-level predictions
    for orig_idx, y_t, p_val, y_p in zip(
        df_test.index, y_test_icu, test_probs, test_preds
    ):
        loio_patient_predictions.append({
            "Source_Index": orig_idx,
            "Left_Out_ICU": test_icu,
            "Fold": fold_idx,
            "y_true": int(y_t),
            "y_proba": float(p_val),
            "y_pred": int(y_p),
            "Optimal_Threshold": float(best_threshold_fold),
            "Calibration_Method": CALIBRATION_METHOD,
        })

    # Save ICU-level results
    fold_metrics = calculate_all_metrics(
        y_test_icu,
        test_probs,
        best_threshold_fold,
    )
    tn, fp, fn, tp = confusion_matrix(
        y_test_icu,
        test_preds,
        labels=[0, 1],
    ).ravel()

    loio_fold_results.append({
        "Result_Level": "Left-out ICU",
        "Left_Out_ICU": test_icu,
        "Fold": fold_idx,
        "Test_N": test_n,
        "Test_Events": test_events,
        "Positive_Rate_%": round(100 * test_events / test_n, 2) if test_n else 0.0,
        "Optimal_Threshold": round(best_threshold_fold, 4),
        "Calibration_Method": CALIBRATION_METHOD,
        "Best_C": fold_result["final_best_params"]["classifier__C"],
        "Best_Class_Weight": str(
            fold_result["final_best_params"]["classifier__class_weight"]
        ),
        "Accuracy": fold_metrics["Accuracy"],
        "Precision_PPV": fold_metrics["Precision_PPV"],
        "Recall_Sensitivity": fold_metrics["Recall_Sensitivity"],
        "Specificity": fold_metrics["Specificity"],
        "NPV": fold_metrics["NPV"],
        "F1_score": fold_metrics["F1_score"],
        "ROC_AUC": fold_metrics["ROC_AUC"],
        "PR_AUC": fold_metrics["PR_AUC"],
        "Brier": fold_metrics["Brier"],
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

loio_results_df = pd.DataFrame(loio_fold_results)
loio_predictions_df = pd.DataFrame(loio_patient_predictions)

print("\n=== LOIO Cross-Validation Complete ===")


In [ ]:
# ==============================================================================
# Cell 29: LOIO Performance Summary and Export
# ==============================================================================
# Reports pooled patient-level results and macro-average results across ICUs.

# Pooled metrics across all left-out ICU predictions
pooled_y_true = loio_predictions_df["y_true"].astype(int).to_numpy()
pooled_y_proba = loio_predictions_df["y_proba"].astype(float).to_numpy()
pooled_y_pred = loio_predictions_df["y_pred"].astype(int).to_numpy()

pooled_tn, pooled_fp, pooled_fn, pooled_tp = confusion_matrix(
    pooled_y_true,
    pooled_y_pred,
    labels=[0, 1],
).ravel()

pooled_auc = roc_auc_score(pooled_y_true, pooled_y_proba)
pooled_prauc = average_precision_score(pooled_y_true, pooled_y_proba)
pooled_brier = brier_score_loss(pooled_y_true, pooled_y_proba)

pooled_metrics = {
    "Result_Level": "Pooled left-out ICUs",
    "Left_Out_ICU": "Pooled overall",
    "Fold": "All",
    "Test_N": len(pooled_y_true),
    "Test_Events": int(pooled_y_true.sum()),
    "Positive_Rate_%": round(100 * pooled_y_true.mean(), 2),
    "Optimal_Threshold": "Fold-specific",
    "Calibration_Method": CALIBRATION_METHOD,
    "Best_C": "Fold-specific",
    "Best_Class_Weight": "Fold-specific",
    "Accuracy": accuracy_score(pooled_y_true, pooled_y_pred),
    "Precision_PPV": precision_score(pooled_y_true, pooled_y_pred, zero_division=0),
    "Recall_Sensitivity": recall_score(
        pooled_y_true,
        pooled_y_pred,
        zero_division=0,
    ),
    "Specificity": (
        pooled_tn / (pooled_tn + pooled_fp)
        if (pooled_tn + pooled_fp) else np.nan
    ),
    "NPV": (
        pooled_tn / (pooled_tn + pooled_fn)
        if (pooled_tn + pooled_fn) else np.nan
    ),
    "F1_score": f1_score(pooled_y_true, pooled_y_pred, zero_division=0),
    "ROC_AUC": pooled_auc,
    "PR_AUC": pooled_prauc,
    "Brier": pooled_brier,
    "TN": int(pooled_tn),
    "FP": int(pooled_fp),
    "FN": int(pooled_fn),
    "TP": int(pooled_tp),
}

# Macro-average metrics across left-out ICUs with 95% t confidence intervals
macro_metric_cols = [
    "Accuracy",
    "Precision_PPV",
    "Recall_Sensitivity",
    "Specificity",
    "NPV",
    "F1_score",
    "ROC_AUC",
    "PR_AUC",
    "Brier",
]

macro_rows = []

for metric in macro_metric_cols:
    values = pd.to_numeric(
        loio_results_df[metric],
        errors="coerce",
    ).dropna().to_numpy()

    n_valid = len(values)

    if n_valid > 1:
        mean_value = float(values.mean())
        sd_value = float(values.std(ddof=1))
        se_value = sd_value / np.sqrt(n_valid)
        t_critical = stats.t.ppf(0.975, df=n_valid - 1)
        ci_low = mean_value - t_critical * se_value
        ci_high = mean_value + t_critical * se_value
    else:
        mean_value = float(values[0]) if n_valid == 1 else np.nan
        sd_value = ci_low = ci_high = np.nan

    macro_rows.append({
        "Metric": metric,
        "Macro_Mean": mean_value,
        "Macro_SD": sd_value,
        "N_Valid_ICUs": n_valid,
        "95%_CI_Lower": ci_low,
        "95%_CI_Upper": ci_high,
        "95%_CI_String": (
            f"{ci_low:.3f}–{ci_high:.3f}"
            if pd.notna(ci_low) else "NA"
        ),
    })

macro_summary_df = pd.DataFrame(macro_rows)
complete_summary_df = pd.concat(
    [loio_results_df, pd.DataFrame([pooled_metrics])],
    ignore_index=True,
)

# Save tables
loio_results_df.to_csv(
    LOIO_TABLE_DIR / "loio_per_icu_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)
loio_predictions_df.to_csv(
    LOIO_TABLE_DIR / "loio_patient_level_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)
macro_summary_df.to_csv(
    LOIO_TABLE_DIR / "loio_macro_summary_95ci.csv",
    index=False,
    encoding="utf-8-sig",
)
complete_summary_df.to_csv(
    LOIO_TABLE_DIR / "loio_complete_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Save analysis details
summary_metadata = {
    "analysis": "Leave-One-ICU-Out cross-validation within one clinical center",
    "target": target,
    "icu_column": group_column,
    "n_icus": n_icus,
    "n_total_patients": len(loio_predictions_df),
    "feature_selection": f"Fold-specific SHAP Top-{TOP_K_FEATURES}",
    "scoring_metric": LOIO_SCORING,
    "calibration": f"Development validation set ({CALIBRATION_METHOD})",
    "threshold_tuning": "Fold-specific development validation set; F1-maximizing threshold",
    "seed": int(SEED),
    "pooled_metrics": {
        key: value
        for key, value in pooled_metrics.items()
        if isinstance(value, (int, float))
    },
}

with open(
    LOIO_MODEL_INFO_DIR / "loio_summary_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary_metadata, file, indent=4, ensure_ascii=False, default=str)

print("=== LOIO Summary Tables Exported ===")

display_cols = [
    "Result_Level",
    "Left_Out_ICU",
    "Test_N",
    "Test_Events",
    "Accuracy",
    "Recall_Sensitivity",
    "Specificity",
    "F1_score",
    "ROC_AUC",
    "PR_AUC",
    "Brier",
]

display(complete_summary_df[display_cols].round(3))


In [ ]:
# ==============================================================================
# Cell 30: LOIO Figure 1 — Discrimination Performance (ROC & PR Curves)
# ==============================================================================
# Shows discrimination across ICUs during LOIO validation.
# Bold lines show the pooled results, while faint lines show results for each held-out ICUs..

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13.5, 6.0))

# Panel A: ROC Curves (Individual ICUs and Pooled)
for icu in unique_icus:
    sub = loio_predictions_df[loio_predictions_df["Left_Out_ICU"] == icu]
    y_t = sub["y_true"].to_numpy()
    y_p = sub["y_proba"].to_numpy()
    if len(np.unique(y_t)) == 2:
        fpr, tpr, _ = roc_curve(y_t, y_p)
        ax_roc.plot(
            fpr, tpr,
            color=STYLE_PALETTE["neutral"],
            alpha=0.35,
            linewidth=1.0,
        )

pooled_fpr, pooled_tpr, _ = roc_curve(pooled_y_true, pooled_y_proba)
ax_roc.plot(
    pooled_fpr, pooled_tpr,
    color=STYLE_PALETTE["primary"],
    linewidth=2.4,
    label=f"Pooled Overall (AUROC = {pooled_auc:.3f})",
)
ax_roc.plot(
    [0, 1], [0, 1],
    linestyle=":",
    color="#555555",
    linewidth=1.1,
    label="Chance (0.50)",
)
ax_roc.set(
    xlim=(-0.02, 1.02),
    ylim=(-0.02, 1.02),
    xlabel="1 - Specificity (False Positive Rate)",
    ylabel="Sensitivity (True Positive Rate)",
    title="Panel A: LOIO ROC Curves Across ICUs",
)
ax_roc.legend(loc="lower right", frameon=True)

# Panel B: Precision-Recall Curves (Individual ICUs and Pooled)
baseline_prevalence = float(np.mean(pooled_y_true))

for icu in unique_icus:
    sub = loio_predictions_df[loio_predictions_df["Left_Out_ICU"] == icu]
    y_t = sub["y_true"].to_numpy()
    y_p = sub["y_proba"].to_numpy()
    if len(np.unique(y_t)) == 2:
        prec, rec, _ = precision_recall_curve(y_t, y_p)
        ax_pr.plot(
            rec, prec,
            color=STYLE_PALETTE["neutral"],
            alpha=0.35,
            linewidth=1.0,
        )

pooled_prec, pooled_rec, _ = precision_recall_curve(pooled_y_true, pooled_y_proba)
ax_pr.plot(
    pooled_rec, pooled_prec,
    color=STYLE_PALETTE["test"],
    linewidth=2.4,
    label=f"Pooled Overall (AUPRC = {pooled_prauc:.3f})",
)
ax_pr.axhline(
    baseline_prevalence,
    linestyle=":",
    color="#555555",
    linewidth=1.1,
    label=f"Prevalence Baseline ({baseline_prevalence*100:.1f}%)",
)
ax_pr.set(
    xlim=(-0.02, 1.02),
    ylim=(-0.02, 1.02),
    xlabel="Recall (Sensitivity)",
    ylabel="Precision (PPV)",
    title="Panel B: LOIO PR Curves Across ICUs",
)
ax_pr.legend(loc="upper right", frameon=True)

fig.tight_layout()

# Save figures
fig_stem_roc_pr = "loio_pooled_and_icu_roc_pr"
for ext in ["png", "pdf", "svg"]:
    fig.savefig(
        LOIO_FIG_DIR / f"{fig_stem_roc_pr}.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight",
    )

plt.show()
plt.close(fig)

# Simple automatic summary
roc_pr_interp_text = f"""
LOIO DISCRIMINATION PERFORMANCE — SUMMARY
Cross-ICU validation ({n_icus} ICUs):

- Pooled AUROC: {_fmt(pooled_auc)}
- Pooled AUPRC: {_fmt(pooled_prauc)}
- Baseline CLABSI Prevalence: {_fmt_pct(baseline_prevalence)}

Key Takeaways:
- Faint gray curves show performance in each left-out ICU.
- Solid bold curves show pooled performance across all patients.
- Shows how well the model distinguishes between risk levels when applied to an unseen ICU.
""".strip()

save_interpretation(
    interpretation_text=roc_pr_interp_text,
    interpretation_dict={
        "figure": fig_stem_roc_pr,
        "n_icus": n_icus,
        "pooled_roc_auc": pooled_auc,
        "pooled_pr_auc": pooled_prauc,
        "baseline_prevalence": baseline_prevalence,
    },
    filename_stem=f"{fig_stem_roc_pr}_interpretation",
    output_dir=LOIO_MODEL_INFO_DIR,
)

print(roc_pr_interp_text)


In [ ]:
# ==============================================================================
# Cell 31: LOIO Pooled Calibration
# ==============================================================================
# Shows pooled calibration across left-out ICUs.

p_y_true = loio_predictions_df["y_true"].astype(int).to_numpy()
p_y_proba = loio_predictions_df["y_proba"].astype(float).to_numpy()
n_loio_icus = loio_predictions_df["Left_Out_ICU"].nunique()
n_calibration_bins = CONTRACT_CONFIG["n_bins_calibration"]

# Pooled quantile-based calibration data
loio_quantiles = pd.qcut(
    pd.Series(p_y_proba).rank(method="first"),
    q=n_calibration_bins,
    labels=False,
    duplicates="drop",
)

loio_cal_df = (
    pd.DataFrame({
        "y_true": p_y_true,
        "y_proba": p_y_proba,
        "bin": loio_quantiles.to_numpy(),
    })
    .groupby("bin", observed=False)
    .agg(
        mean_pred=("y_proba", "mean"),
        obs_rate=("y_true", "mean"),
        count=("y_true", "count"),
    )
    .reset_index(drop=True)
)

loio_brier = brier_score_loss(p_y_true, p_y_proba)
loio_cal_dev = np.mean(
    np.abs(loio_cal_df["mean_pred"] - loio_cal_df["obs_rate"])
)

set_publication_style()
plt.close("all")

fig_loio_cal, ax = plt.subplots(figsize=(6.5, 5.5))

# Faint lines show calibration in each left-out ICU
for icu in loio_predictions_df["Left_Out_ICU"].unique():
    sub = loio_predictions_df[loio_predictions_df["Left_Out_ICU"] == icu]

    if sub["y_true"].nunique() == 2 and len(sub) >= 15:
        icu_bins = pd.qcut(
            sub["y_proba"].rank(method="first"),
            q=4,
            labels=False,
            duplicates="drop",
        )
        icu_cal_df = (
            sub.groupby(icu_bins, observed=False)
            .agg(
                mean_pred=("y_proba", "mean"),
                obs_rate=("y_true", "mean"),
            )
        )
        ax.plot(
            icu_cal_df["mean_pred"],
            icu_cal_df["obs_rate"],
            color=STYLE_PALETTE["neutral"],
            alpha=0.35,
            linewidth=1.0,
        )

# Bold line shows pooled calibration
ax.plot(
    loio_cal_df["mean_pred"],
    loio_cal_df["obs_rate"],
    marker="s",
    color=STYLE_PALETTE["danger"],
    linewidth=2.0,
    label=f"Pooled Results (Brier = {loio_brier:.3f})",
)
ax.plot(
    [0, 1], [0, 1],
    linestyle=":",
    color="#555555",
    linewidth=1.2,
    label="Ideal",
)

ax.set(
    xlabel="Mean Predicted Probability",
    ylabel="Observed Event Rate",
    title=f"LOIO Calibration Across ICUs",
)
ax.legend(loc="upper left", frameon=True)

add_interpretation_box(
    ax,
    (
        f"Pooled results from {n_loio_icus} left-out ICUs "
        f"(n = {len(p_y_true)}).\n"
        "Quantile bins help keep similar numbers of patients in each group."
    ),
    x=0.03,
    y=0.05,
    fontsize=8.0,
)

fig_loio_cal.tight_layout()

# Save figure
fig_stem_loio_cal = "loio_pooled_calibration"

for ext in ["png", "svg", "pdf"]:
    fig_loio_cal.savefig(
        LOIO_FIG_DIR / f"{fig_stem_loio_cal}.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight",
    )

display(fig_loio_cal)
plt.close(fig_loio_cal)

loio_cal_text = f"""
LOIO POOLED CALIBRATION — SUMMARY
Cross-ICU validation ({n_loio_icus} ICUs):

- Bins: {n_calibration_bins} quantile bins
- Pooled patients: {len(p_y_true)}
- Pooled Brier score: {_fmt(loio_brier)}
- Mean absolute calibration gap: {_fmt(loio_cal_dev)}

Assessment:
This plot shows whether predicted risks remain well calibrated
when the model is applied to a left-out ICU.
""".strip()

save_interpretation(
    interpretation_text=loio_cal_text,
    interpretation_dict={
        "evaluation": "LOIO pooled calibration",
        "n_icus": int(n_loio_icus),
        "n_samples": len(p_y_true),
        "n_bins": int(n_calibration_bins),
        "brier_score": float(loio_brier),
        "mean_calibration_gap": float(loio_cal_dev),
    },
    filename_stem=f"{fig_stem_loio_cal}_interpretation",
    output_dir=LOIO_MODEL_INFO_DIR,
)

print(loio_cal_text)


In [ ]:
# ==============================================================================
# Cell 32: LOIO Figure 3 — ICU-Level ROC-AUC Forest Plot
# ==============================================================================
# Compares ROC-AUC values across left-out ICUs.

valid_icu_df = (
    loio_results_df
    .dropna(subset=["ROC_AUC"])
    .sort_values("ROC_AUC")
    .reset_index(drop=True)
)

macro_auc_row = macro_summary_df.loc[
    macro_summary_df["Metric"] == "ROC_AUC"
].iloc[0]

macro_mean_auc = macro_auc_row["Macro_Mean"]
macro_ci_low = macro_auc_row["95%_CI_Lower"]
macro_ci_high = macro_auc_row["95%_CI_Upper"]

fig, ax_forest = plt.subplots(
    figsize=(8.5, max(5.0, len(valid_icu_df) * 0.45 + 2.0))
)

y_positions = np.arange(len(valid_icu_df))

# Individual ICU AUCs
ax_forest.scatter(
    valid_icu_df["ROC_AUC"],
    y_positions,
    color=STYLE_PALETTE["primary"],
    s=65,
    zorder=3,
    label="Left-out ICU AUC",
)

for idx, row in valid_icu_df.iterrows():
    ax_forest.text(
        0.02,
        idx,
        (
            f"{row['Left_Out_ICU']} "
            f"(N = {int(row['Test_N'])}, Events = {int(row['Test_Events'])})"
        ),
        va="center",
        fontsize=8.5,
    )

# Macro-average and pooled results
summary_y_macro = len(valid_icu_df) + 0.6
summary_y_pooled = len(valid_icu_df) + 1.4

ax_forest.errorbar(
    macro_mean_auc,
    summary_y_macro,
    xerr=[
        [macro_mean_auc - macro_ci_low],
        [macro_ci_high - macro_mean_auc],
    ],
    fmt="D",
    color=STYLE_PALETTE["secondary"],
    markersize=7,
    capsize=5,
    elinewidth=1.6,
    zorder=4,
    label="Macro Average (95% CI)",
)
ax_forest.text(
    0.02,
    summary_y_macro,
    f"Macro Average ({macro_auc_row['95%_CI_String']})",
    va="center",
    fontweight="bold",
    fontsize=9,
)

ax_forest.scatter(
    pooled_auc,
    summary_y_pooled,
    marker="X",
    color=STYLE_PALETTE["danger"],
    s=90,
    zorder=4,
    label=f"Pooled Overall (AUC = {pooled_auc:.3f})",
)
ax_forest.text(
    0.02,
    summary_y_pooled,
    f"Pooled Overall (AUC = {pooled_auc:.3f})",
    va="center",
    fontweight="bold",
    fontsize=9,
)

ax_forest.axvline(
    0.5,
    linestyle=":",
    color="#777777",
    linewidth=1.0,
)
ax_forest.axvline(
    macro_mean_auc,
    linestyle="--",
    color=STYLE_PALETTE["secondary"],
    alpha=0.5,
    linewidth=1.1,
)

ax_forest.set(
    yticks=[],
    xlabel="Area Under the ROC Curve (ROC-AUC)",
    title="LOIO Discrimination Across ICUs",
    xlim=(0.0, 1.05),
    ylim=(-0.8, summary_y_pooled + 0.8),
)
ax_forest.legend(loc="lower right", frameon=True)

fig.tight_layout()

# Save figure
fig_stem_forest = "loio_icu_auc_forest_plot"

for ext in ["png", "pdf", "svg"]:
    fig.savefig(
        LOIO_FIG_DIR / f"{fig_stem_forest}.{ext}",
        dpi=600 if ext == "png" else None,
        bbox_inches="tight",
    )

forest_interp_text = f"""
LOIO ICU-LEVEL ROC-AUC — SUMMARY
Cross-ICU validation within one clinical center:

- ICUs with valid ROC-AUC: {len(valid_icu_df)}
- Macro-average ROC-AUC: {_fmt(macro_mean_auc)}
- Macro 95% CI: {macro_auc_row["95%_CI_String"]}
- Pooled ROC-AUC: {_fmt(pooled_auc)}

Assessment:
Each blue point shows performance in one left-out ICU.
Differences between ICUs may reflect variation in patient mix,
event frequency, and ICU-specific clinical practice.
""".strip()

forest_interp_dict = {
    "figure": fig_stem_forest,
    "evaluation": "LOIO within one clinical center",
    "n_evaluable_icus": len(valid_icu_df),
    "macro_mean_auc": float(macro_mean_auc),
    "macro_95ci": [float(macro_ci_low), float(macro_ci_high)],
    "pooled_auc": float(pooled_auc),
}

save_interpretation(
    interpretation_text=forest_interp_text,
    interpretation_dict=forest_interp_dict,
    filename_stem=f"{fig_stem_forest}_interpretation",
    output_dir=LOIO_MODEL_INFO_DIR,
)

plt.show()
plt.close(fig)
